## 1. Imports
Everything the notebook needs, in one place (no repeated/duplicate import cells).

In [1]:
import json
import difflib
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher, get_close_matches

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading


c:\Users\HP\anaconda3\envs\tf_env\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
c:\Users\HP\anaconda3\envs\tf_env\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Users\HP\anaconda3\envs\tf_env\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical 

## 2. Define data sources (paths listed once, files read exactly once below)

In [2]:
files = [
    r"C:\Users\HP\Downloads\files (18)\neu_phone_directory.json",
    r"C:\Users\HP\Downloads\files (18)\neu_admissions_academic.json",
    r"C:\Users\HP\Downloads\files (18)\neu_campus_facilities.json",
    r"C:\Users\HP\Downloads\files (18)\neu_faculties_departments.json",
    r"C:\Users\HP\Downloads\files (18)\neu_general_info.json",
    r"C:\Users\HP\Downloads\files (18)\neu_hospital_departments.json",
    r"C:\Users\HP\Downloads\files (18)\neu_locations.json",
    r"C:\Users\HP\Downloads\files (18)\neu_qa_dataset.txt",
    r"C:\Users\HP\Downloads\files (18)\neu_research.json",
    r"C:\Users\HP\Downloads\files (18)\README.md",
    r"C:\Users\HP\Downloads\files (18)\level1111.csv"
]

## 3. Load every file ONCE
Builds the `Document` objects for embedding, and caches every parsed JSON file in `RAW_JSON_CACHE` so the doublecheck step further down can reuse it instead of reading the files from disk a second time.

In [3]:
import unicodedata

all_docs = []
ROOM_LOOKUP = {}      # normalized room_number -> room chunk text (exact lookup for short codes like "U08")
LECTURER_INDEX = []   # list of (name_tokens: frozenset, room_text: str, display_name: str)
ROOM_DOCS = []         # every room-type Document, kept in memory for direct keyword
                       # matching in dataset_answer() -- bypasses the FAISS top-k cutoff,
                       # which can bury short/terse room chunks (e.g. "Grand Library
                       # Archives") beneath longer, more generic facility/QA text.

# Academic titles/degree abbreviations to drop when comparing names, so "Prof. Dr. X Y"
# and a question like "where is X Y" both reduce to the same meaningful token set.
_TITLE_WORDS = {
    "prof", "dr", "assoc", "asst", "eng", "msc", "mse", "ma", "med", "phd",
    "doc", "aras", "gor", "dt", "mr", "mrs", "ms",
}

def _fold_diacritics(s: str) -> str:
    """ASCII-fold a string so Turkish characters compare equal regardless of how they're
    typed: 'Akgün' / 'akgun' / 'AKGUN' all fold to 'akgun'. Turkish dotted/dotless I
    (İ/ı) aren't handled correctly by plain NFKD+lower, so they're special-cased first."""
    s = s.replace("İ", "I").replace("ı", "i")
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower()

def _name_tokens(raw: str) -> frozenset:
    """Extract meaningful, diacritic-folded name tokens from a lecturer string or a
    question, dropping punctuation and academic titles/degrees."""
    cleaned = re.sub(r"[.,]", " ", str(raw).replace("-", ""))
    words = [_fold_diacritics(w) for w in cleaned.split()]
    return frozenset(w for w in words if w and w not in _TITLE_WORDS)

# The level_u.csv room_number column stores values like "Room U08" / "Room 6"
# (the word "Room" is baked into the value itself, not just the room's name).
_ROOM_PREFIX_RE = re.compile(
    r"^(?:room|office|lab|hall|classroom)\.?\s*(?:no\.?|number)?\s*[:\-]?\s*",
    re.IGNORECASE,
)

def _normalize_room_key(val) -> str:
    """Normalize a room_number value (from the CSV OR extracted from a user's question)
    into the SAME lookup key, regardless of which side it came from. Handles:
      1. If ANY row in the room_number column is blank, pandas silently upcasts the
         WHOLE column to float64, turning e.g. 6 into 6.0 -> "6.0". We collapse that
         back to "6" so purely-numeric rooms ("Room 6") still match.
      2. A leading "Room"/"Office"/"Lab"/etc. word, since the CSV's room_number values
         already include it (e.g. "Room U08"), but questions get parsed with that word
         stripped off separately -- so both sides need the SAME stripping to line up.
      3. Spaces/dashes/case differences ("U-08" vs "u08" vs "U 08") are stripped so
         they all resolve to the same key.
    """
    s = str(val).strip().lower()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    s = _ROOM_PREFIX_RE.sub("", s)
    return s.replace("-", "").replace(" ", "")

# Function to load JSON files
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Function to load text files
def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def parse_qa_dataset(text: str):
    """Parses the `Q: ... / A: ...` pairs in neu_qa_dataset.txt into a list of
    dicts. Pure text parsing - no LLM, no embedding call here."""
    lines = [l for l in text.split("\n") if not l.strip().startswith("#")]
    text = "\n".join(lines)
    blocks = re.split(r"\n(?=Q:\s)", text)
    pairs = []
    for b in blocks:
        b = b.strip()
        if not b.startswith("Q:"):
            continue
        m = re.match(r"Q:\s*(.*?)\nA:\s*(.*)", b, re.DOTALL)
        if m:
            pairs.append({"question": m.group(1).strip(), "answer": m.group(2).strip()})
    return pairs

for file in files:

    # ==========================
    # JSON FILES
    # ==========================
    if file.endswith(".json"):
        data = load_json(file)

        # Faculties dataset
        if isinstance(data, dict) and "faculties" in data:

            for faculty in data["faculties"]:

                departments = "\n".join(
                    f"- {d}" for d in faculty.get("departments", [])
                )

                degree_types = ", ".join(
                    faculty.get("degree_types", [])
                )

                text = f"""
This document is about the {faculty['name']}.

Faculty Name: {faculty['name']}

Departments:
{departments}

Degree Types:
{degree_types}

Website:
{faculty.get('website','')}
"""

                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "faculty_name": faculty["name"],
                            "type": "faculty"
                        }
                    )
                )

        # Hospital departments dataset -> one readable document per department
        elif isinstance(data, dict) and "departments" in data and isinstance(data["departments"], dict):

            for dept_id, d in data["departments"].items():

                hours_parts = [
                 f"{k.replace('_', '–').title()}: {v}"
                 for k, v in d.get("hours", {}).items()
                ]
                hours_text = "; ".join(hours_parts) if hours_parts else "Not specified"
                
                extra_lines = []
                if d.get("sub_departments"):
                    extra_lines.append("Sub-departments: " + ", ".join(d["sub_departments"]))
                if d.get("sub_clinics"):
                    extra_lines.append("Sub-clinics: " + ", ".join(d["sub_clinics"]))
                if d.get("equipment"):
                    extra_lines.append("Equipment: " + ", ".join(d["equipment"]))
                if d.get("sub_units"):
                    unit_names = [u.get("name", "") for u in d["sub_units"] if isinstance(u, dict)]
                    extra_lines.append("Sub-units: " + ", ".join(unit_names))
                extra_text = "\n".join(extra_lines)

                text = f"""
This document describes a department at NEU Hospital.

Department: {d.get('name_en', dept_id)}
Floor: {d.get('floor', '')}
Block: {d.get('block', '')}
Hours: {hours_text}
Facilities: {', '.join(d.get('facilities', []))}
{extra_text}
Notes: {d.get('notes', '')}
"""

                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "department_name": d.get("name_en", dept_id),
                            "type": "hospital_department",
                            "floor": d.get("floor", ""),
                            "block": d.get("block", ""),
                        }
                    )
                )
        
            # --- NEW: general hospital info from _metadata -----------------------
            meta = data.get("_metadata", {})
            structure = meta.get("hospital_structure", {})
            contact = meta.get("contact", {})
            booking = meta.get("appointment_booking", {})

            info_lines = [
                "This document describes general information about NEU Hospital.",
                f"Total area: {structure.get('total_area_sqm', '')} sqm",
                f"Blocks: {', '.join(structure.get('blocks', []))}",
                f"Floors (East): {structure.get('floors_east', '')}",
                f"Floors (West): {structure.get('floors_west', '')}",
                f"Total patient rooms: {structure.get('total_patient_rooms', '')}",
                f"VIP rooms: {structure.get('vip_rooms', '')}",
                f"Max bed capacity: {structure.get('max_bed_capacity', '')}",
                f"Phone: {contact.get('phone', '')}",
                f"Website: {contact.get('website', '')}",
                f"Address: {contact.get('address', '')}",
                f"Appointment booking website: {booking.get('website', '')}",
                f"Appointment booking phone: {booking.get('phone', '')}",
            ]
            all_docs.append(
                Document(
                    page_content="\n".join(info_lines),
                    metadata={
                        "source": file,
                        "type": "hospital_info",
                    }
                )
            )
        
        
         # --- NEW: Research centers dataset -> one readable document per center ---
        elif isinstance(data, dict) and "research" in data:
            research = data["research"]
            for center in research.get("notable_centers", []):
                extra_lines = []
                if center.get("description"):
                    extra_lines.append(f"Description: {center['description']}")
                if center.get("location"):
                    extra_lines.append(f"Location: {center['location']}")
                if center.get("website"):
                    extra_lines.append(f"Website: {center['website']}")
                if center.get("established"):
                    extra_lines.append(f"Established: {center['established']}")
                if center.get("curator_founder"):
                    extra_lines.append(f"Curator/Founder: {center['curator_founder']}")
                if center.get("research_areas"):
                    extra_lines.append("Research areas: " + ", ".join(center["research_areas"]))
                if center.get("notable_projects"):
                    extra_lines.append("Notable projects: " + ", ".join(center["notable_projects"]))
                extra_text = "\n".join(extra_lines)

                text = f"""
This document describes a NEU research center.

Name: {center.get('name', 'Unknown')}
{extra_text}
"""
                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "type": "research_center",
                            "center_name": center.get("name", "Unknown"),
                        }
                    )
                )

        # Campus locations dataset -> one readable document per building/location
        elif isinstance(data, dict) and ("academic_buildings" in data or "key_locations" in data):

            entries = list(data.get("academic_buildings", [])) + list(data.get("key_locations", []))

            for loc in entries:

                text_parts = [
                    "This document describes a location on the NEU campus.",
                    f"Name: {loc.get('name', '')}",
                    f"Description: {loc.get('location_description', '')}",
                    f"Directions: {loc.get('directions', '')}",
                ]
                if loc.get("hours"):
                    text_parts.append(f"Hours: {loc['hours']}")
                phone = loc.get("phone") or loc.get("phone_extension")
                if phone:
                    text_parts.append(f"Phone: {phone}")
                if loc.get("departments_housed"):
                    text_parts.append("Departments housed here: " + ", ".join(loc["departments_housed"]))

                all_docs.append(
                    Document(
                        page_content="\n".join(text_parts),
                        metadata={
                            "source": file,
                            "location_name": loc.get("name", ""),
                            "type": "location"
                        }
                    )
                )
        
        # Campus locations dataset -> one readable document per building/location
        elif isinstance(data, dict) and ("academic_buildings" in data or "key_locations" in data):
            entries = list(data.get("academic_buildings", [])) + list(data.get("key_locations", []))
            for loc in entries:
                text_parts = [
                    "This document describes a location on the NEU campus.",
                    f"Name: {loc.get('name', '')}",
                    f"Description: {loc.get('location_description', '')}",
                    f"Directions: {loc.get('directions', '')}",
                ]
                if loc.get("hours"):
                    text_parts.append(f"Hours: {loc['hours']}")
                phone = loc.get("phone") or loc.get("phone_extension")
                if phone:
                    text_parts.append(f"Phone: {phone}")
                if loc.get("departments_housed"):
                    text_parts.append("Departments housed here: " + ", ".join(loc["departments_housed"]))

                all_docs.append(
                    Document(
                        page_content="\n".join(text_parts),
                        metadata={
                            "source": file,
                            "location_name": loc.get("name", ""),
                            "type": "location"
                        }
                    )
                )

        # --- NEW: Research centers dataset -> one readable document per center ---
        elif isinstance(data, dict) and "research" in data:
            research = data["research"]
            for center in research.get("notable_centers", []):
                extra_lines = []
                if center.get("description"):
                    extra_lines.append(f"Description: {center['description']}")
                if center.get("location"):
                    extra_lines.append(f"Location: {center['location']}")
                if center.get("website"):
                    extra_lines.append(f"Website: {center['website']}")
                if center.get("established"):
                    extra_lines.append(f"Established: {center['established']}")
                if center.get("curator_founder"):
                    extra_lines.append(f"Curator/Founder: {center['curator_founder']}")
                if center.get("research_areas"):
                    extra_lines.append("Research areas: " + ", ".join(center["research_areas"]))
                if center.get("notable_projects"):
                    extra_lines.append("Notable projects: " + ", ".join(center["notable_projects"]))
                extra_text = "\n".join(extra_lines)

                text = f"""
This document describes a NEU research center.

Name: {center.get('name', 'Unknown')}
{extra_text}
"""
                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "type": "research_center",
                            "center_name": center.get("name", "Unknown"),
                        }
                    )
                )
        

        # Every other JSON file (admissions, general_info, campus_facilities, research, ...):
        # flatten into (readable label -> verbatim value) leaf pairs instead of dumping the
        # whole file as one raw JSON blob. Each leaf becomes its own small, readable document,
        # so a doublecheck hit returns a clean sentence instead of a jagged JSON fragment.
        else:

            def _flatten_json(obj, prefix=""):
                items = []
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        label = f"{prefix} {k.replace('_', ' ')}".strip()
                        if isinstance(v, dict):
                            items.extend(_flatten_json(v, label))
                        elif isinstance(v, list):
                            if all(isinstance(e, (str, int, float)) for e in v):
                                items.append((label, "; ".join(str(e) for e in v)))
                            else:
                                for entry in v:
                                    sub_prefix = label
                                    if isinstance(entry, dict):
                                        name = entry.get("name") or entry.get("title")
                                        if name:
                                            sub_prefix = f"{label} {name}"
                                    items.extend(_flatten_json(entry, sub_prefix))
                        else:
                            items.append((label, str(v)))
                return items

            for label, value in _flatten_json(data):
                all_docs.append(
                    Document(
                        page_content=f"{label}: {value}",
                        metadata={
                            "source": file,
                            "type": "field"
                        }
                    )
                )

    # ==========================
    # CSV FILES
    # ==========================
    elif file.endswith(".csv"):

         df = pd.read_csv(file, encoding="cp1252", dtype=str, keep_default_na=False)

         # Group by room_number + building + floor, not room_number alone.
         # The CSV reuses generic room_number values like "Room Kitchen" for
         # multiple physically distinct rooms on different floors -- grouping by
         # room_number alone silently merges them into one, discarding all but
         # the first floor's location data.
         df["_group_key"] = (
             df["room_number"].astype(str).str.strip() + " | " +
             df["building"].astype(str).str.strip() + " | " +
             df["floor"].astype(str).str.strip()
         )
         grouped = df.groupby("_group_key", dropna=False)
         def _clean(v):
             """Turn missing/NaN/blank CSV cells into None instead of the literal
             string "nan" (which pandas produces when an empty cell is f-string'd
             directly), so blank fields can be skipped instead of printed."""
             if v is None:
                 return None
             if isinstance(v, float) and pd.isna(v):
                 return None
             s = str(v).strip()
             if not s or s.lower() == "nan":
                 return None
             return s

         for _group_key, group in grouped:
           room_number = group.iloc[0]["room_number"]  # get the REAL room_number back, not the composite key

           # One clean "Name (Position)" line per lecturer -- skip rows with no name,
           # and skip the position entirely if it's missing rather than printing "nan".
           lecturers = []
           for _, row in group.iterrows():
               name = _clean(row.get("Lecturers"))
               if not name:
                   continue
               pos = _clean(row.get("Position/Title"))
               lecturers.append(f"{name} ({pos})" if pos else name)

           building        = _clean(group.iloc[0].get("building"))
           floor           = _clean(group.iloc[0].get("floor"))
           room_name       = _clean(group.iloc[0].get("room_name"))
           room_type       = _clean(group.iloc[0].get("room_type"))
           location_desc   = _clean(group.iloc[0].get("location_description"))

           # Build the text as a list of lines and only include fields that actually
           # have a value -- no more "location description: nan" / blank trailing lines.
           lines = ["This document describes a room at the NEU campus.", ""]
           if building:      lines.append(f"Building: {building}")
           if floor:         lines.append(f"Floor: {floor}")
           lines.append(f"Room Number: {room_number}")
           if room_name:     lines.append(f"Room Name: {room_name}")
           if room_type:     lines.append(f"Room Type: {room_type}")
           if location_desc: lines.append(f"Location description: {location_desc}")
           if lecturers:
               lines.append("Lecturers:")
               lines.extend(f"- {l}" for l in lecturers)
           text = "\n".join(lines)
           # `text` above stays structured -- it's what gets embedded into FAISS, and a
           # structured layout is fine (even good) for that. But it's a bad *chat* answer.
           # Build a natural-language sentence version, used only for the exact-match
           # shortcuts below, so what the user actually reads looks like a reply, not a
           # database dump.
           room_label = str(room_number)
           if room_name and room_name.strip().lower() != room_label.lower():
               room_label += f' ("{room_name}")'

           where_bits = []
           if building: where_bits.append(f"the {building.title()}")
           if floor:    where_bits.append(floor)
           where_clause = " in " + ", ".join(where_bits) if where_bits else ""

           room_sentence = f"{room_label} is located{where_clause}."
           if lecturers:
               if len(lecturers) == 1:
                   room_sentence += f" {lecturers[0]} is based there."
               else:
                   room_sentence += " The people based there are: " + ", ".join(lecturers) + "."

           room_doc = Document(
                page_content=text,
                metadata={
                    "source": file,
                    "type": "room",
                    "building": building or "",
                    "floor": floor or "",
                    "room_number": str(room_number),
                    "room_name": room_name or "",
                    "room_type": room_type or "",
                    "location description": location_desc or "",
                    "lecturers": lecturers,
                   }
            )
           all_docs.append(room_doc)
           ROOM_DOCS.append(room_doc)   # NEW -- direct handle for keyword matching

           # Index this room under a normalized key so exact codes -- whether
           # alphanumeric ("U08") or purely numeric ("6") -- can be looked up directly,
           # bypassing semantic embedding search (unreliable for short exact codes).
           room_key = _normalize_room_key(room_number)
           if room_key:
               ROOM_LOOKUP[room_key] = room_sentence

           # Index each lecturer in this room by diacritic-folded name tokens, so a
           # question like "where is Yigit Akgün" can be matched directly against the
           # CSV data -- this bypasses both the semantic embedding search AND the
           # ASCII-only tokenizer used later for the confidence check, neither of which
           # handles Turkish characters (ü/ğ/ş/ı/ö/ç) correctly.
           for _, row in group.iterrows():
               lect_raw = row.get("Lecturers", "")
               if pd.isna(lect_raw) or not str(lect_raw).strip():
                   continue
               name_tokens = _name_tokens(lect_raw)
               if len(name_tokens) < 2:  # need at least first+last name to be identifiable
                   continue

               display_name = str(lect_raw).strip()
               pos = _clean(row.get("Position/Title"))
               person_lead = f"{display_name} ({pos})" if pos else display_name

               colleagues = [l for l in lecturers if not l.startswith(display_name)]
               person_sentence = f"{person_lead} is based in {room_label}, located{where_clause}."
               if colleagues:
                   person_sentence += " Also based there: " + ", ".join(colleagues) + "."

               LECTURER_INDEX.append((name_tokens, person_sentence, display_name))

    # ==========================
    # TXT / MD FILES
    # ==========================
    elif file.endswith(".txt") or file.endswith(".md"):

        text = load_txt(file)

        # The curated Q&A dataset gets folded in as ONE Document per (question,
        # answer) pair -- embedding just the question -- so it lands in the
        # SAME FAISS store as everything else and can be retrieved through the
        # SAME top-K retriever, instead of being embedded a second time later
        # outside FAISS.
        if "qa_dataset" in file.lower():
            qa_pairs = parse_qa_dataset(text)
            for qp in qa_pairs:
                all_docs.append(
                    Document(
                        page_content=qp["question"],
                        metadata={
                            "source": file,
                            "type": "qa_pair",
                            "answer": qp["answer"],
                        }
                    )
                )
        else:
            all_docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file,
                        "type": "text"
                    }
                )
            )

    else:
        print(f"[skip] Unsupported file type: {file}")

print("Total documents (incl. QA pairs):", len(all_docs))


Total documents (incl. QA pairs): 1367


In [4]:
# --- DIAGNOSTIC: inspect the CSV room-matching dataset ---------------------------
# Run this AFTER the "Load every file ONCE" cell (and after restarting the kernel /
# re-running the notebook top-to-bottom, so ROOM_LOOKUP reflects the latest code).

print("Total rooms indexed in ROOM_LOOKUP:", len(ROOM_LOOKUP))
print("\nFirst 20 keys in ROOM_LOOKUP:")
for k in list(ROOM_LOOKUP.keys())[:20]:
    print(" -", repr(k))

print("\nIs 'u08' present as a key?", "u08" in ROOM_LOOKUP)

# Show the raw room_number column straight from level_u.csv, so you can see exactly
# what values/dtype pandas actually parsed (this is the #1 cause of mismatches --
# e.g. dtype, stray whitespace, different capitalization, or a different column name).
_csv_path = [f for f in files if f.lower().endswith("level_u.csv")]
if _csv_path:
    _df_check = pd.read_csv(_csv_path[0], encoding="cp1252")
    print("\nColumns in level_u.csv:", list(_df_check.columns))
    if "room_number" in _df_check.columns:
        print("room_number dtype:", _df_check["room_number"].dtype)
        print("Unique room_number values (first 30):")
        print(_df_check["room_number"].unique()[:30])
    else:
        print("\n[!] 'room_number' column NOT found in level_u.csv -- this is the bug.")
        print("    The CSV-loading code expects a column literally named 'room_number'.")
else:
    print("level_u.csv not found in `files` list.")


Total rooms indexed in ROOM_LOOKUP: 118

First 20 keys in ROOM_LOOKUP:
 - '10'
 - '101'
 - '102'
 - '103'
 - '104'
 - '105'
 - '106'
 - '107'
 - '108'
 - '109'
 - '11'
 - '110'
 - '111'
 - '112'
 - '113'
 - '114'
 - '115'
 - '117'
 - '118'
 - '119'

Is 'u08' present as a key? True
level_u.csv not found in `files` list.


## 4. Embedding model

In [5]:
embeddings = HuggingFaceEmbeddings( # convert text into numbers
    model_name="sentence-transformers/all-MiniLM-L6-v2" # Vector Database Implementation
)

## 5. Chunking

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))

Total chunks: 1414


In [7]:
import statistics as st, collections
print("docs by type:", collections.Counter(d.metadata.get("type") for d in all_docs))
print("docs by file:", collections.Counter(d.metadata.get("source","").split("\\")[-1] for d in all_docs))
L = [len(c.page_content) for c in chunks]
print("chunk len mean/median/max:", round(st.mean(L),1), st.median(L), max(L))
print("ROOM_DOCS:", len(ROOM_DOCS), "| ROOM_LOOKUP:", len(ROOM_LOOKUP), "| LECTURER_INDEX:", len(LECTURER_INDEX))

docs by type: Counter({'field': 892, 'qa_pair': 219, 'room': 125, 'hospital_department': 55, 'location': 47, 'faculty': 20, 'research_center': 7, 'hospital_info': 1, 'text': 1})
docs by file: Counter({'neu_campus_facilities.json': 482, 'neu_phone_directory.json': 232, 'neu_qa_dataset.txt': 219, 'level1111.csv': 125, 'neu_admissions_academic.json': 103, 'neu_general_info.json': 75, 'neu_hospital_departments.json': 56, 'neu_locations.json': 47, 'neu_faculties_departments.json': 20, 'neu_research.json': 7, 'README.md': 1})
chunk len mean/median/max: 121.4 76.0 498
ROOM_DOCS: 125 | ROOM_LOOKUP: 118 | LECTURER_INDEX: 113


## 6. Vectorize — build the FAISS vector database (once)

In [8]:
vectorstore = FAISS.from_documents(chunks, embeddings) # chunks: These are your small pieces of text after splitting, embeddings: It converts text → numbers (vectors).

## 7. Retriever (connects to the vector database)

In [9]:
retriever = vectorstore.as_retriever( # retriever : Finds and returns the most relevant pieces of information from your knowledge base based on a user question.
    search_type="similarity",         # This connects your chatbot to the FAISS database.
    search_kwargs={"k": 5} 
)

## 8. Prompt template for RAG

In [10]:
SYSTEM_PROMPT = """\
You are NEU Assistant, a friendly and helpful AI for Near East University (NEU) in Nicosia, Cyprus.

Your personality:
- Friendly, helpful, and professional
- Respond concisely but completely

Your responsibilities:
1. ACADEMIC       → Answer questions about faculties, departments, programmes, admissions, and courses.
2. LOCATION       → Provide building names, campus map references, or directions when asked where something is.
3. GENERAL INFO   → Cover research centres, hospital, facilities, events, and university history.
4. ANY QUESTION   → If the question is not about NEU, answer it naturally and helpfully from your own general knowledge. Do NOT say you can't help or redirect to NEU.
"""
GENERAL_FALLBACK_TEMPLATE = PromptTemplate(
    input_variables=["question"],
    partial_variables={"system_prompt": SYSTEM_PROMPT},
    template="""{system_prompt}

The user's question below was NOT found in NEU's internal knowledge base, so you have no specific data to draw on for it. Answer using your own general knowledge instead — but always remember: "NEU" in this conversation means Near East University in Nicosia, Northern Cyprus. It does NOT mean Northeastern University in Boston, nor any German town/place starting with "Neu". If the question is about NEU itself (its location, how to get there, campus life, etc.) and you don't know the specific answer, say so honestly rather than guessing or substituting an unrelated "NEU".

User question: {question}

Answer:""",
)

RAG_TEMPLATE = """\
{system_prompt}

--- Relevant Context from NEU Knowledge Base ---
{context}
------------------------------------------------

User question: {question}

Answer directly (no greeting, cite your source at the end):"""

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    partial_variables={"system_prompt": SYSTEM_PROMPT},
    template=RAG_TEMPLATE,
)

print("Prompt template ready. ✅")

Prompt template ready. ✅


## 9. LLM

In [11]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6Lhgj348r0-Vz_s0fBkW7X2RUVop6vLDLO0MOu-SZNi2g"

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

## 10. Connect the vector database to the LLM (the RAG chain)

In [12]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("QA chain ready.")

QA chain ready.


## 11. Doublecheck — is the top retrieved chunk good enough to answer with, no LLM?

Matches the architecture exactly: the SAME `vectorstore` and top-K retrieval used for the LLM path is reused here. `dataset_answer()` takes the best-scoring retrieved chunk and asks one question — is it similar enough to the user's question to trust as-is?

- **YES** -> return that chunk's own text (its stored `answer` if it's a curated Q&A chunk, otherwise the chunk's own content) as the final answer. Zero LLM calls.
- **NO** -> return `None`, and the caller falls back to `qa_chain` (retrieved chunks + question -> LLM) further up.

No separate raw-JSON/`difflib` lookups anymore — one retrieval, one confidence check.

In [13]:
_STOPWORDS = {"of", "the", "and", "in", "at", "is", "are", "for", "a", "an", "to", "on", "s",
              "what", "who", "why", "how", "when", "where", "does", "do", "can", "i", "you", "me",
              "it", "this", "that", "be", "about",
              "neu", "university", "campus"}

# Ordinary question words that can also be surnames/first names in the CSV
# (e.g. "Can"). A match on ONLY one of these must never count as a person lookup.
_QUESTION_FILLER = {
    "how", "can", "could", "would", "should", "will", "may", "might", "must", "shall",
    "i", "we", "you", "me", "my", "need", "want", "to", "go", "get", "reach", "locate",
    "find", "where", "what", "who", "which", "when", "why", "is", "are", "am", "was",
    "were", "be", "the", "a", "an", "of", "in", "at", "on", "for", "please", "tell",
    "show", "give", "about", "does", "do", "did", "have", "has", "long", "young", "bay",
}

# Short tokens are dropped by default (too many spurious matches otherwise),
# but a handful of 2-letter facility codes are meaningful on their own and
# would otherwise vanish silently -- e.g. "wc" (Room "bathroom"'s own
# room_name is literally "WC").
_SHORT_TOKEN_ALLOW = {"wc"}

def _tokenize(s: str):
    return [
        w for w in re.findall(r"[a-z0-9]+", s.lower())
        if w not in _STOPWORDS
        and (len(w) >= 3 or any(char.isdigit() for char in w) or w in _SHORT_TOKEN_ALLOW)
    ]

# Generic/structural words that show up as filler inside tons of unrelated room
# names ("Technical Room", "Print Room", "Meeting Room"...) -- long enough to
# clear a naive length check, but useless for identifying WHICH room something
# is. Excluded from the substring side of the check; exact-token matches still
# work normally (e.g. a room whose room_type is literally "Bathroom").
_FUZZY_EXCLUDE = {"room", "office", "building", "floor", "level", "hall", "lab", "classroom"}

# Synonym groups for facility vocabulary that differs across buildings in the
# dataset itself -- International's rooms are typed "Bathroom" directly,
# Innovation's equivalent rooms are typed "Restroom (WC) for MEN/WOMEN".
# Same facility, but zero tokens in common, so no amount of exact or
# substring matching can ever bridge the two on its own.
_SYNONYM_GROUPS = [
    {"bathroom", "restroom", "toilet", "washroom", "lavatory"},
    # add more groups here as you discover other naming mismatches
]
_SYNONYMS = {}
for _group in _SYNONYM_GROUPS:
    for _word in _group:
        _SYNONYMS[_word] = _group - {_word}

def _expand_synonyms(tokens: set) -> set:
    """Add every known synonym for each token already present, so a query for
    "bathroom" also matches content tokenized as "restroom" -- and so does the
    plural "bathrooms", which won't be a literal key in _SYNONYMS."""
    expanded = set(tokens)
    for t in tokens:
        expanded |= _SYNONYMS.get(t, set())
        if t.endswith("s"):
            singular = t[:-1]
            expanded |= _SYNONYMS.get(singular, set())
    return expanded

def _fuzzy_overlap(set_a: set, set_b: set, min_len: int = 4) -> bool:
    """True on an exact shared token, OR when one token is a substring of the
    other. Generic/structural words (_FUZZY_EXCLUDE) are stripped from BOTH
    sides first, so a shared word like "room" alone can never count as a
    match -- only meaningfully specific words (e.g. "technical", "print") can."""
    meaningful_a = {a for a in set_a if a not in _FUZZY_EXCLUDE}
    meaningful_b = {b for b in set_b if b not in _FUZZY_EXCLUDE}

    if meaningful_a & meaningful_b:
        return True

    for a in meaningful_a:
        if len(a) < min_len:
            continue
        for b in meaningful_b:
            if len(b) < min_len:
                continue
            if a in b or b in a:
                return True
    return False

CHUNK_MATCH_THRESHOLD = 0.30

def _clean_label(doc) -> str:
    label = (
        doc.metadata.get("faculty_name")
        or doc.metadata.get("department_name")
        or doc.metadata.get("location_name")
        or doc.metadata.get("source", "NEU Knowledge Base")
    )
    if "\\" in label or "/" in label:
        label = label.replace("\\", "/").split("/")[-1]
    return label

def _cos_sim_from_l2(l2_dist: float) -> float:
    return 1 - (l2_dist ** 2) / 2

_ALL_BUILDING_TOKENS = set()
_ALL_BUILDING_NAME_TOKENS = []   # one frozenset per DISTINCT building name, for precise
                                  # identification -- loose word-overlap breaks the moment
                                  # two buildings share a generic word, e.g. "Innovation
                                  # Building" vs "International Building" both containing "building".
_seen_building_names = set()
for _d in all_docs:
    _b = _d.metadata.get("building")
    if _b:
        _ALL_BUILDING_TOKENS |= set(_tokenize(_b))
        if _b not in _seen_building_names:
            _seen_building_names.add(_b)
            _ALL_BUILDING_NAME_TOKENS.append(frozenset(_tokenize(_b)))

_GENERIC_BUILDING_WORDS = {"building"}

def _named_buildings_in(tokens: set) -> list:
    """Every known building whose MEANINGFUL name-token-set (i.e. excluding
    generic filler words like "building") is contained in `tokens`. This lets
    "innovation" alone match "Innovation Building" without requiring the word
    "building" to be typed -- while still requiring the full, specific part of
    the name (so "international" alone can't match "Innovation Building")."""
    matches = []
    for bt in _ALL_BUILDING_NAME_TOKENS:
        meaningful = bt - _GENERIC_BUILDING_WORDS
        if meaningful and meaningful <= tokens:
            matches.append(bt)
    return matches

# Vocabulary of real words that appear in your data, so typo'd query words can
# be nudged toward something that actually exists in the corpus.
_VOCAB = set()
for d in all_docs:
    _VOCAB.update(_tokenize(d.page_content))
    for key in ("department_name", "room_name", "room_type", "building",
                "faculty_name", "location_name", "center_name"):
        val = d.metadata.get(key)
        if val:
            _VOCAB.update(_tokenize(str(val)))

def _correct_query(text: str, cutoff: float = 0.6) -> str:
    """Best-effort spelling correction against dataset vocabulary. A badly
    garbled word (e.g. 'emregcney') otherwise embeds too far from anything
    relevant to clear CHUNK_MATCH_THRESHOLD -- correcting it first fixes that
    at the source, rather than patching the token-overlap filter downstream."""
    corrected_words = []
    for w in text.split():
        w_clean = re.sub(r"[^\w]", "", w.lower())
        if not w_clean or len(w_clean) < 4 or w_clean in _VOCAB:
            corrected_words.append(w)
            continue
        close = difflib.get_close_matches(w_clean, _VOCAB, n=1, cutoff=cutoff)
        corrected_words.append(close[0] if close else w)
    return " ".join(corrected_words)


def dataset_answer(question: str, k: int = 30):
    # --- NEW: detect "how to go to X" / directions-seeking questions -------------
    _DIRECTIONS_RE = re.compile(
        r"\b("
        r"how\s+(?:to|do\s+i|can\s+i)\s+(?:go|get|reach)|"
        r"directions?\s+to|"
        r"way\s+to\s+(?:get|go)|"
        r"(?:i|we)\s+(?:need|want)\s+to\s+(?:go|get|reach)\s+to|"
        r"(?:take|guide)\s+(?:me|us)\s+to"
        r")\b",
        re.IGNORECASE,
    )
    is_directions_question = bool(_DIRECTIONS_RE.search(question))

    # --- NEW: "hospital info" shortcut --------------------------------------
    _HOSPITAL_INFO_RE = re.compile(
        r"\bhospital\s+info(?:rmation)?\b|"
        r"\babout\s+(?:the\s+)?(?:neu\s+)?hospital\b|"
        r"\bgeneral\s+(?:info(?:rmation)?\s+)?(?:about\s+)?(?:the\s+)?hospital\b|"
        r"\bwhere\s+is\s+(?:the\s+)?(?:neu\s+)?hospital\b|"
        r"\bhospital\s+(?:location|address)\b|"
        r"\blocation\s+of\s+(?:the\s+)?(?:neu\s+)?hospital\b",
        re.IGNORECASE,
    )

    if _HOSPITAL_INFO_RE.search(question):
        info_docs = [
            d for d in all_docs
            if d.metadata.get("type") == "hospital_info"
        ]

        if info_docs:
            return {
                "answer": info_docs[0].page_content.strip(),
                "source": "neu_hospital_departments.json",
                "method": "full-listing (hospital_info)",
                "needs_phrasing": False,
            }

    # --- Exact room-code shortcut -----------------------------------------------
    _ROOM_CONTEXT_RE = re.compile(
        r"\b(?:room|office|lab|hall|classroom)\.?\s*"
        r"(?:no\.?|number)?\s*[:\-]?\s*"
        r"([a-z]{0,3}-?\d{1,4}[a-z]?)\b",
        re.IGNORECASE,
    )

    candidate_codes = [
        m.group(1)
        for m in _ROOM_CONTEXT_RE.finditer(question)
    ]

    candidate_codes += re.findall(
        r"[a-z]{1,3}-?\d{1,4}[a-z]?",
        question.lower()
    )

    for raw_code in candidate_codes:
        code_key = _normalize_room_key(raw_code)

        if code_key in ROOM_LOOKUP:
            return {
                "answer": ROOM_LOOKUP[code_key],
                "source": "level1111.csv",
                "method": f"exact-room-match ({raw_code})",
                "needs_phrasing": False,
            }

    # --- NEW: "list all departments" shortcut -----------------------------------
    # A generic "what departments are there" question can never reliably surface
    # all 55 hospital departments through FAISS top-k + keyword overlap -- vector
    # search returns nearest neighbors, not full category membership. Detect this
    # intent and pull every hospital_department doc directly from memory instead.
    _LIST_DEPTS_RE = re.compile(
        r"\b(all|every|list)\b.*\bdepartments?\b|"
        r"\bwhat\s+(?:are\s+the\s+)?departments\b|"
        r"\bdepartments?\s+(?:are\s+)?(?:in|at)\s+(?:the\s+)?hospital\b|"
        r"\bhospital\s+departments?\b",
        re.IGNORECASE,
    )

    if _LIST_DEPTS_RE.search(question):
        dept_docs = [
            d for d in all_docs
            if d.metadata.get("type") == "hospital_department"
        ]

        if dept_docs:
            lines = [
                f"- {d.metadata.get('department_name', 'Unknown')}"
                for d in dept_docs
            ]

            answer_text = (
                f"NEU Hospital has {len(dept_docs)} departments:\n\n"
                + "\n".join(lines)
            )

            return {
                "answer": answer_text,
                "source": "neu_hospital_departments.json",
                "method": (
                    f"full-listing "
                    f"(all {len(dept_docs)} hospital departments)"
                ),
                "needs_phrasing": False,
            }

    # --- NEW: "floor X departments/rooms" shortcut -------------------------------
    # Floor is spelled inconsistently across datasets ("1st Floor" in hospital JSON
    # vs "First Floor" in CSV rooms) -- normalize both sides to a canonical digit
    # so "first floor" reliably matches "1st Floor" and vice versa. This is a
    # filter/enumeration query, same as "list all departments" -- FAISS top-k +
    # token overlap can't guarantee completeness here either.
    _ORDINAL_MAP = {
        "ground": "0", "0th": "0",
        "first": "1", "1st": "1",
        "second": "2", "2nd": "2",
        "third": "3", "3rd": "3",
        "fourth": "4", "4th": "4",
        "fifth": "5", "5th": "5",
        "sixth": "6", "6th": "6",
        "seventh": "7", "7th": "7",
        "eighth": "8", "8th": "8",
        "ninth": "9", "9th": "9",
    }

    def _canonical_floor(floor_str: str):
        floor_str = (floor_str or "").lower()

        for word, digit in _ORDINAL_MAP.items():
            if word in floor_str:
                return digit

        return None

    _FLOOR_QUERY_RE = re.compile(
        r"\b(ground|first|1st|second|2nd|third|3rd|fourth|4th|fifth|5th|"
        r"sixth|6th|seventh|7th|eighth|8th|ninth|9th)\s+floor\b.*"
        r"\b(departments?|rooms?)\b",
        re.IGNORECASE,
    )

    floor_match = _FLOOR_QUERY_RE.search(question)

    if floor_match:
        target_floor = _canonical_floor(floor_match.group(1))

        wants_hospital = "hospital" in question.lower()

        wants_rooms = any(
            w in question.lower()
            for w in ("room", "building", "office", "classroom")
        )

        allowed_types = []

        if wants_hospital or not wants_rooms:
            allowed_types.append("hospital_department")

        if wants_rooms or not wants_hospital:
            allowed_types.append("room")

        floor_docs = [
            d for d in all_docs
            if d.metadata.get("type") in allowed_types
            and _canonical_floor(
                d.metadata.get("floor", "")
            ) == target_floor
        ]

        if floor_docs:
            lines = []

            for d in floor_docs:
                if d.metadata.get("type") == "hospital_department":
                    lines.append(
                        f"- {d.metadata.get('department_name')} "
                        f"(NEU Hospital, {d.metadata.get('block', '')})"
                    )
                else:
                    label = (
                        d.metadata.get("room_name")
                        or d.metadata.get("room_number", "")
                    )

                    lines.append(
                        f"- {label} "
                        f"({d.metadata.get('building', '')})"
                    )

            answer_text = (
                "Departments/rooms on this floor:\n\n"
                + "\n".join(lines)
            )

            return {
                "answer": answer_text,
                "source": "combined dataset",
                "method": (
                    f"full-listing "
                    f"(floor match, {len(floor_docs)} results)"
                ),
                "needs_phrasing": False,
            }
    
    # NEW: "how many faculties" -> count them from the data
    _COUNT_FACULTIES_RE = re.compile(
        r"\b(how\s+many|number\s+of|total|count)\b.*\bfacult(?:y|ies)\b(?!\s+of\b)",
        re.IGNORECASE,
    )

    if _COUNT_FACULTIES_RE.search(question):
        faculty_docs = [d for d in all_docs if d.metadata.get("type") == "faculty"]
        if faculty_docs:
            names = "\n".join(f"- {d.metadata.get('faculty_name')}" for d in faculty_docs)
            return {
                "answer": f"NEU has {len(faculty_docs)} faculties:\n\n{names}",
                "source": "neu_faculties_departments.json",
                "method": f"faculty-count ({len(faculty_docs)})",
                "needs_phrasing": False,
            }        

    # --- NEW: "faculties and departments" shortcut -------------------------------
    _LIST_FACULTIES_RE = re.compile(
        r"\b(all|every|list)\b.*\bfacult(?:y|ies)\b|"
        r"\bwhat\s+facult(?:y|ies)\b|"
        r"\bfacult(?:y|ies)\s*(?:and|&)\s*departments?\b|"
        r"\bfacult(?:y|ies)\s*&\s*departments?\b",
        re.IGNORECASE,
    )

    if _LIST_FACULTIES_RE.search(question):
        faculty_docs = [
            d for d in all_docs
            if d.metadata.get("type") == "faculty"
        ]

        if faculty_docs:
            combined = "\n\n---\n\n".join(
                d.page_content.strip()
                for d in faculty_docs
            )

            answer_text = (
                f"NEU has {len(faculty_docs)} faculties:\n\n"
                f"{combined}"
            )

            return {
                "answer": answer_text,
                "source": "neu_faculties_departments.json",
                "method": (
                    f"full-listing "
                    f"(all {len(faculty_docs)} faculties)"
                ),
                "needs_phrasing": False,
            }

    # --- NEW: "campus locations" shortcut -----------------------------------
    _LIST_LOCATIONS_RE = re.compile(
        r"\b(all|every|list)\b.*\blocations?\b|"
        r"\bwhat\s+locations?\b|"
        r"\bcampus\s*locations?\b|"
        r"\bwhere\s+is\s+(?:the\s+)?campus\b",
        re.IGNORECASE,
    )

    if _LIST_LOCATIONS_RE.search(question):
        location_docs = [
            d for d in all_docs
            if d.metadata.get("type") == "location"
        ]

        if location_docs:
            combined = "\n\n---\n\n".join(
                d.page_content.strip()
                for d in location_docs
            )

            answer_text = (
                f"NEU has {len(location_docs)} key campus locations:\n\n"
                f"{combined}"
            )

            return {
                "answer": answer_text,
                "source": "neu_campus_facilities.json",
                "method": (
                    f"full-listing "
                    f"(all {len(location_docs)} campus locations)"
                ),
                "needs_phrasing": False,
            }

    # --- NEW: "research centers" shortcut -------------------------------------
    _LIST_RESEARCH_RE = re.compile(
        r"\b(all|every|list)\b.*\bresearch\s*centers?\b|"
        r"\bwhat\s+research\s*centers?\b|"
        r"\bresearch\s*centers?\b",
        re.IGNORECASE,
    )

    if _LIST_RESEARCH_RE.search(question):
        research_docs = [
            d for d in all_docs
            if d.metadata.get("type") == "research_center"
        ]

        if research_docs:
            combined = "\n\n---\n\n".join(
                d.page_content.strip()
                for d in research_docs
            )

            answer_text = (
                f"NEU has 28 research centers in total. The dataset provides named "
                f"details for {len(research_docs)} of them:\n\n{combined}"
            )

            return {
                "answer": answer_text,
                "source": "neu_research.json",
                "method": (
                    f"full-listing "
                    f"({len(research_docs)} of 28 research centers named in dataset)"
                ),
                "needs_phrasing": False,
            }

    # --- NEW: "departments of a specific faculty" shortcut -----------------------
    _FACULTY_ALIASES = {
        "ai": "artificial intelligence",
        "it": "information technology",
    }

    def _expand_faculty_aliases(text: str) -> str:
        words = text.lower().split()

        return " ".join(
            _FACULTY_ALIASES.get(w, w)
            for w in words
        )

    _FACULTY_TRIGGER_WORDS = {
        "department",
        "departments",
        "faculty",
        "faculties",
        "facility",
        "facilities"
    }

    _q_words_raw = question.lower().split()

    def _is_faculty_trigger(word: str) -> bool:
        # Exact/fuzzy spelling match (catches typos like "departnemt").
        if difflib.get_close_matches(
            word,
            _FACULTY_TRIGGER_WORDS,
            n=1,
            cutoff=0.7
        ):
            return True

        # Prefix match for truncations/abbreviations (catches "dep", "dept", "fac").
        if len(word) >= 3 and any(
            t.startswith(word)
            for t in _FACULTY_TRIGGER_WORDS
        ):
            return True

        return False

    _has_faculty_trigger = any(
        _is_faculty_trigger(w)
        for w in _q_words_raw
    )

    if _has_faculty_trigger:
        expanded_q = _expand_faculty_aliases(question)

        q_words = (
            set(_tokenize(expanded_q))
            | set(
                w for w in expanded_q.split()
                if len(w) <= 2
            )
        )

        best_doc = None
        best_score = 0
        
        # NEW: generic words must not count as evidence of a specific faculty
        _GENERIC = {"faculty", "faculties", "department", "departments",
                    "facility", "facilities"}
        q_specific = q_words - _GENERIC

        for d in all_docs:
            if d.metadata.get("type") != "faculty":
                continue

            faculty_words = set(
                _tokenize(
                    d.metadata.get("faculty_name", "")
                )
            ) - _GENERIC                       # changed

            overlap = q_specific & faculty_words
            score = len(overlap)

            if score == 0:
                for qw in q_specific:
                    close = difflib.get_close_matches(
                        qw,
                        faculty_words,
                        n=1,
                        cutoff=0.8
                    )

                    if close:
                        score = 1
                        break

            if score > best_score:
                best_score = score
                best_doc = d

        if best_doc and best_score >= 1:
            return {
                "answer": best_doc.page_content.strip(),
                "source": "neu_faculties_departments.json",
                "method": (
                    f"faculty-match "
                    f"({best_doc.metadata.get('faculty_name')}, "
                    f"score={best_score})"
                ),
                "needs_phrasing": False,
            }

    # --- Exact / partial lecturer-name shortcut ----------------------------------
    q_name_tokens = frozenset() if is_directions_question else _name_tokens(question)

    if q_name_tokens:
        best_score = 0
        matches_at_best = []

        for name_tokens, room_text, display_name in LECTURER_INDEX:
            overlap = name_tokens & q_name_tokens

            if not overlap:
                continue
            
            # NEW: a lone match on a common question word ("can") is not a person lookup
            if len(overlap) == 1 and next(iter(overlap)) in _QUESTION_FILLER:
                continue

            score = len(overlap)

            if score > best_score:
                best_score, matches_at_best = (
                    score,
                    [(room_text, display_name)]
                )

            elif score == best_score:
                matches_at_best.append(
                    (room_text, display_name)
                )

        # Full name match (>=2 shared tokens) -> confident, no LLM.
        if matches_at_best and best_score >= 2:
            distinct_rooms = {}

            for room_text, display_name in matches_at_best:
                distinct_rooms.setdefault(
                    room_text,
                    display_name
                )

            if len(distinct_rooms) == 1:
                room_text, display_name = next(
                    iter(distinct_rooms.items())
                )

                return {
                    "answer": room_text,
                    "source": "level1111.csv",
                    "method": (
                        f"exact-lecturer-match "
                        f"({display_name})"
                    ),
                    "needs_phrasing": False,
                }

            else:
                combined = "\n\n---\n\n".join(
                    distinct_rooms.keys()
                )

                return {
                    "answer": (
                        "This name is associated with more than one "
                        f"location:\n\n{combined}"
                    ),
                    "source": "level1111.csv",
                    "method": "exact-lecturer-match (multiple rooms)",
                    "needs_phrasing": False,
                }

        # Partial match (e.g. first name only) -> still useful, less confident.
        elif matches_at_best and best_score == 1:
            # Dedupe by ROOM first (same as the full-name branch), so a person who
            # holds two different roles/rooms under the identical name still shows
            # both, instead of silently collapsing to whichever row was seen first.
            distinct_rooms = {}

            for room_text, display_name in matches_at_best:
                distinct_rooms.setdefault(
                    room_text,
                    display_name
                )

            if len(distinct_rooms) == 1:
                room_text, display_name = next(
                    iter(distinct_rooms.items())
                )

                return {
                    "answer": room_text,
                    "source": "level1111.csv",
                    "method": (
                        f"partial-lecturer-match "
                        f"({display_name})"
                    ),
                    "needs_phrasing": False,
                }

            unique_names = set(
                distinct_rooms.values()
            )

            if len({_name_tokens(n) for n in unique_names}) == 1:
                # Same person, genuinely in more than one room (multiple roles) ->
                # list all of them, just like the full-name-match case does.
                display_name = next(
                    iter(unique_names)
                )

                combined = "\n\n---\n\n".join(
                    distinct_rooms.keys()
                )

                return {
                    "answer": (
                        "This name is associated with more than one "
                        f"location:\n\n{combined}"
                    ),
                    "source": "level1111.csv",
                    "method": (
                        f"partial-lecturer-match "
                        f"({display_name}, multiple rooms)"
                    ),
                    "needs_phrasing": False,
                }

            else:
                # Different people who happen to share a first name -> ask for the
                # full name instead of guessing.
                names = ", ".join(
                    sorted(unique_names)
                )

                return {
                    "answer": (
                        f"I found more than one person matching that name: "
                        f"{names}. "
                        "Could you give me the full name so I can find the right one?"
                    ),
                    "source": "level1111.csv",
                    "method": "partial-lecturer-match (ambiguous)",
                    "needs_phrasing": False,
                }

    # --- NEW: keyword-matched room shortcut (bypasses the FAISS top-k cutoff) ---
    # Only runs for non-direction questions that explicitly ask about a room,
    # office, bathroom, WC, etc. Directions questions must go through the
    # location-only semantic retrieval below.

    q_tokens_kw = set(_tokenize(question))
    named_buildings_kw = _named_buildings_in(q_tokens_kw)

    if named_buildings_kw and not is_directions_question:

        _EXPLICIT_ROOM_WORDS = {
            "room", "office", "lab", "hall", "classroom",
            "bathroom", "restroom", "toilet", "washroom",
            "lavatory", "wc"
        }

        has_explicit_room_request = any(
            word in q_tokens_kw
            for word in _EXPLICIT_ROOM_WORDS
        )

        if has_explicit_room_request:

            meaningful_q_tokens = _expand_synonyms(
                q_tokens_kw - _ALL_BUILDING_TOKENS
            )

            keyword_room_matches = []

            for doc in ROOM_DOCS:
                room_name_tokens = set(
                    _tokenize(
                        doc.metadata.get("room_name", "")
                    )
                )

                room_type_tokens = set(
                    _tokenize(
                        doc.metadata.get("room_type", "")
                    )
                )

                if not _fuzzy_overlap(
                    meaningful_q_tokens,
                    room_name_tokens | room_type_tokens
                ):
                    continue

                room_building_tokens = frozenset(
                    _tokenize(
                        doc.metadata.get("building", "")
                    )
                )

                if room_building_tokens in named_buildings_kw:
                    keyword_room_matches.append(doc)

            if keyword_room_matches:
                lines = []

                for doc in keyword_room_matches:
                    m = doc.metadata

                    room_num = m.get("room_number", "")
                    room_name = m.get("room_name", "")
                    building = m.get("building", "")
                    floor = m.get("floor", "")

                    room_num_str = str(room_num)

                    label = (
                        room_num_str
                        if room_num_str.lower().startswith("room")
                        else f"Room {room_num_str}"
                    )

                    if not room_num:
                        label = "A room"

                    if (
                        room_name
                        and room_name.strip().lower()
                        != str(room_num).strip().lower()
                    ):
                        label += f' ("{room_name}")'

                    where_bits = []

                    if building:
                        where_bits.append(
                            f"the {building.title()}"
                        )

                    if floor:
                        where_bits.append(floor)

                    where_clause = (
                        " in " + ", ".join(where_bits)
                        if where_bits
                        else ""
                    )

                    lines.append(
                        f"- {label}, located{where_clause}."
                    )

                plural = len(keyword_room_matches) > 1

                answer_text = (
                    "There is more than one match:\n\n"
                    if plural
                    else ""
                ) + "\n".join(lines)

                return {
                    "answer": answer_text,
                    "source": "level1111.csv",
                    "method": (
                        f"keyword-room-match "
                        f"({len(keyword_room_matches)} result"
                        f"{'s' if plural else ''})"
                    ),
                    "needs_phrasing": False,
                }

    # --- Semantic retrieval: collect ALL genuinely matching chunks, not just #1 ---
    search_query = _correct_query(question)

    hits = vectorstore.similarity_search_with_score(
        search_query,
        k=k
    )

    if not hits:
        return None

    q_tokens = set(
        _tokenize(search_query)
    )

    matches = []

    for doc, l2_dist in hits:
        cos_sim = _cos_sim_from_l2(l2_dist)

        if cos_sim < CHUNK_MATCH_THRESHOLD:
            break

        if is_directions_question and doc.metadata.get("type") != "location":
            # "How to go to X" can only be genuinely answered by a location entry
            # that actually has directions content -- rooms, research centers,
            # faculty listings, etc. can never answer this, no matter how many
            # words they happen to share with the question.
            continue

        compare_text = doc.metadata.get(
            "question",
            doc.page_content
        )

        structural_tokens = set(
            _tokenize(
                f"{doc.metadata.get('building','')} "
                f"{doc.metadata.get('floor','')} "
                f"room office"
            )
        )

        content_tokens = (
            set(_tokenize(compare_text))
            - structural_tokens
        )

        shared = q_tokens & content_tokens

        if doc.metadata.get("type") == "room":
            # A room matches only if the question shares a word with its NAME or
            # TYPE specifically -- but building-name vocabulary ("international",
            # "innovation", "building") is stripped out first, since those words
            # can coincidentally appear inside an unrelated room's name (e.g. an
            # "International Development Office") without the room having
            # anything to do with the building the user is asking about.

            room_name_tokens = set(
                _tokenize(
                    doc.metadata.get("room_name", "")
                )
            )

            room_type_tokens = set(
                _tokenize(
                    doc.metadata.get("room_type", "")
                )
            )

            meaningful_q_tokens = _expand_synonyms(
                q_tokens - _ALL_BUILDING_TOKENS
            )

            if not _fuzzy_overlap(
                meaningful_q_tokens,
                room_name_tokens | room_type_tokens
            ):
                continue

            # If the question names a specific building, the room must actually be in it
            # -- matched by FULL building name, not by any single shared word.
            named_buildings = _named_buildings_in(q_tokens)

            if named_buildings:
                room_building_tokens = frozenset(
                    _tokenize(
                        doc.metadata.get("building", "")
                    )
                )

                if room_building_tokens not in named_buildings:
                    continue

        else:
            # Non-room chunks (JSON fields, locations, Q&A) are longer, freer text --
            # a single shared word is too easy to hit by coincidence. Require at
            # least 2 genuinely shared content words.
            if len(shared) < 2:
                continue

        matches.append(
            (doc, cos_sim)
        )

    if not matches:
        return None

    # If the question names a specific building and multiple matches remain,
    # narrow to it.
    def _doc_building(doc):
        return (
            doc.metadata.get("building") or ""
        ).lower()

    mentioned = {
        b
        for b in {
            _doc_building(d)
            for d, _ in matches
        }
        if b and b in question.lower()
    }

    if mentioned and len(matches) > 1:
        narrowed = [
            (d, s)
            for d, s in matches
            if _doc_building(d) in mentioned
        ]

        if narrowed:
            matches = narrowed

    if len(matches) == 1:
        doc, cos_sim = matches[0]

        is_qa_pair = (
            doc.metadata.get("type") == "qa_pair"
        )

        if is_qa_pair:
            return {
                "answer": doc.metadata["answer"],
                "source": _clean_label(doc),
                "method": (
                    f"dataset-match "
                    f"(chunk, score={cos_sim:.2f})"
                ),
                "needs_phrasing": False,
            }

        return {
            "answer": doc.page_content.strip(),
            "source": _clean_label(doc),
            "method": (
                f"dataset-match "
                f"(chunk, score={cos_sim:.2f})"
            ),
            "needs_phrasing": True,
        }

    # Multiple genuine matches -> build the listing directly from metadata when
    # possible (deterministic, no risk of the LLM collapsing distinct rooms into
    # one vague sentence), falling back to LLM-phrased combined context otherwise.
    if all(
        doc.metadata.get("type") == "room"
        for doc, _ in matches
    ):
        lines = []

        for doc, _ in matches:
            m = doc.metadata

            room_num = m.get("room_number", "")
            room_name = m.get("room_name", "")
            building = m.get("building", "")
            floor = m.get("floor", "")

            room_num_str = str(room_num)

            label = (
                room_num_str
                if room_num_str.lower().startswith("room")
                else f"Room {room_num_str}"
            )

            if not room_num:
                label = "A room"

            if (
                room_name
                and room_name.strip().lower()
                != str(room_num).strip().lower()
            ):
                label += f' ("{room_name}")'

            where_bits = []

            if building:
                where_bits.append(
                    f"the {building.title()}"
                )

            if floor:
                where_bits.append(floor)

            where_clause = (
                " in " + ", ".join(where_bits)
                if where_bits
                else ""
            )

            lines.append(
                f"- {label}, located{where_clause}."
            )

        answer_text = (
            "There is more than one match:\n\n"
            + "\n".join(lines)
        )

        sources_combined = ", ".join(
            dict.fromkeys(
                _clean_label(doc)
                for doc, _ in matches
            )
        )

        return {
            "answer": answer_text,
            "source": sources_combined,
            "method": (
                f"dataset-match "
                f"(multiple: {len(matches)} results, itemized)"
            ),
            "needs_phrasing": False,
        }

    # Non-room chunks (mixed types) -> combine as context, let Gemini phrase it.
    combined_context = "\n\n---\n\n".join(
        doc.page_content.strip()
        for doc, _ in matches
    )

    sources_combined = ", ".join(
        dict.fromkeys(
            _clean_label(doc)
            for doc, _ in matches
        )
    )

    return {
        "answer": combined_context,
        "source": sources_combined,
        "method": (
            f"dataset-match "
            f"(multiple: {len(matches)} results)"
        ),
        "needs_phrasing": True,
    }

In [14]:
# ============================================================
# FLOOR-AWARE ANSWERS  (works for ANY building in the dataset)
# Run this as a NEW CELL right after the cell that defines dataset_answer()
# (cell 22) and before ask() is used.
#
# Why it was needed: the old "floor" shortcut inside dataset_answer() only fired
# for questions shaped like "<ordinal> floor ... rooms/departments", ignored the
# building name, and mapped "underground" -> "ground". A question like
# "what first floor at innovation building have" matched nothing, so it fell
# through to the LLM, which made up a generic answer.
# ============================================================
from collections import Counter

_ORD = {
    "ground": "0", "0th": "0",
    "first": "1", "1st": "1", "second": "2", "2nd": "2", "third": "3", "3rd": "3",
    "fourth": "4", "4th": "4", "fifth": "5", "5th": "5", "sixth": "6", "6th": "6",
    "seventh": "7", "7th": "7", "eighth": "8", "8th": "8", "ninth": "9", "9th": "9",
    "tenth": "10", "10th": "10",
}
_FLOOR_LABELS = {"u": "Underground floor", "0": "Ground floor", "1": "First floor",
                 "2": "Second floor", "3": "Third floor", "4": "Fourth floor",
                 "5": "Fifth floor", "6": "Sixth floor", "7": "Seventh floor",
                 "8": "Eighth floor", "9": "Ninth floor", "10": "Tenth floor"}

def _floor_key(text):
    """Canonical floor id from ANY dataset spelling: 'level 1 (first floor)', '1st Floor',
    'First Floor', 'ground floor', 'level_u (underground floor)', 'Underground' ...
    -> 'u' | '0' | '1' | '2' ... (None if no floor found).
    Underground is checked BEFORE ground (it contains the word 'ground')."""
    s = (text or "").lower().replace("_", " ")
    if re.search(r"\b(underground|basement|lower ground|level u)\b", s):
        return "u"
    m = re.search(r"\blevel\s*(\d+)\b", s)
    if m:
        return str(int(m.group(1)))
    for word, digit in _ORD.items():
        if re.search(rf"\b{word}\b", s):
            return digit
    m = re.search(r"\bfloor\s*(\d+)\b|\b(\d+)\s*floor\b", s)
    if m:
        return str(int(m.group(1) or m.group(2)))
    return None

_Q_FLOOR_RE = re.compile(
    r"\b(?:underground|basement|ground|first|second|third|fourth|fifth|sixth|seventh|eighth|"
    r"ninth|tenth|\d{1,2}(?:st|nd|rd|th))\s+(?:floor|level)\b"
    r"|\b(?:floor|level)\s*[-_ ]?(?:\d{1,2}|u)\b"
    r"|\b(?:underground|basement)\b",
    re.IGNORECASE,
)

# words that only phrase the question -- they must not be treated as a room filter
_FLOOR_FILLER = {
    "have", "has", "had", "there", "any", "all", "everything", "anything", "show", "list",
    "give", "tell", "need", "want", "find", "please", "contain", "contains", "include",
    "includes", "inside", "located", "available", "offer", "offers", "exist", "exists",
    "which", "whats", "every", "each", "whole", "entire", "things", "stuff", "room", "rooms", "department", "departments", "dept", "dep", "floor", "level", "hospital", "block", "east", "west",
    "ground", "underground", "basement", "first", "second", "third", "fourth", "fifth",
    "sixth", "seventh", "eighth", "ninth", "tenth",
}

def _natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]

def _kw_matches(kws, *token_sources):
    """True if any query keyword matches (plural-insensitive / substring) a token in the sources."""
    kws = _expand_synonyms(set(kws))
    toks = set()
    for src in token_sources:
        toks |= set(_tokenize(src or ""))
    for k in kws:
        ks = k.rstrip("s")
        for t in toks:
            ts = t.rstrip("s")
            if ks == ts:
                return True
            if ts in _FUZZY_EXCLUDE:      # generic words ("room", "hall"...) never fuzzy-match
                continue
            if (len(ks) >= 3 and ts.startswith(ks)) or \
               (len(ks) >= 4 and len(ts) >= 4 and (ks in ts or ts in ks)):
                return True
    return False

def _clean(s):
    return " ".join(str(s or "").split())

def floor_answer(question: str):
    m = _Q_FLOOR_RE.search(question)
    if not m:
        return None
    target = _floor_key(m.group(0))
    if target is None:
        return None

    q_tokens = set(_tokenize(question))
    named = _named_buildings_in(q_tokens)                  # [] if no building was named
    wants_hospital = "hospital" in question.lower()

    # leftover words = optional filter, e.g. "bathrooms", "labs", "offices"
    kws = (q_tokens - _ALL_BUILDING_TOKENS - _FLOOR_FILLER
           - set(_tokenize(m.group(0))) - {t for t in q_tokens if t.isdigit()})

    # ---- campus rooms (CSV) ----
    rooms = []
    if not (wants_hospital and not named):
        for d in ROOM_DOCS:
            if _floor_key(d.metadata.get("floor", "")) != target:
                continue
            if named and frozenset(_tokenize(d.metadata.get("building", ""))) not in named:
                continue
            if kws and not _kw_matches(kws, d.metadata.get("room_name"), d.metadata.get("room_type")):
                continue
            rooms.append(d)

    # ---- hospital departments (JSON) ----
    depts = []
    if wants_hospital or not named:
        for d in all_docs:
            if d.metadata.get("type") != "hospital_department":
                continue
            if _floor_key(d.metadata.get("floor", "")) != target:
                continue
            if kws and not _kw_matches(kws, d.metadata.get("department_name")):
                continue
            depts.append(d)

    if not rooms and not depts:
        return None            # nothing on that floor -> let the normal pipeline decide

    floor_name = _FLOOR_LABELS.get(target, f"Floor {target}")
    blocks = []

    by_building = {}
    for d in rooms:
        by_building.setdefault(d.metadata.get("building") or "Unknown building", []).append(d)

    for building, docs in by_building.items():
        docs.sort(key=lambda d: _natural_key(d.metadata.get("room_number", "")))
        types = Counter(_clean(d.metadata.get("room_type")) or "Room" for d in docs)
        header = (f"{building.title()} - {floor_name}: {len(docs)} room(s)"
                  + (f" matching '{' '.join(sorted(kws))}'" if kws else ""))
        summary = "Types: " + ", ".join(f"{t} x{c}" for t, c in types.most_common())
        lines = []
        for d in docs:
            md = d.metadata
            num = _clean(md.get("room_number"))
            label = num if num.lower().startswith("room") else f"Room {num}" if num else "Room"
            name = _clean(md.get("room_name"))
            rtype = _clean(md.get("room_type"))
            line = f"- {label}"
            if name and name.lower() != num.lower():
                line += f": {name}"
            if rtype and rtype.lower() not in name.lower():
                line += f" [{rtype}]"
            people = md.get("lecturers") or []
            if people:
                shown = ", ".join(_clean(p) for p in people[:3])
                if len(people) > 3:
                    shown += f" (+{len(people) - 3} more)"
                line += f" - {shown}"
            lines.append(line)
        blocks.append(header + "\n" + summary + "\n\n" + "\n".join(lines))

    if depts:
        lines = [f"- {_clean(d.metadata.get('department_name'))} ({_clean(d.metadata.get('block'))})"
                 for d in depts]
        blocks.append(f"NEU Hospital - {floor_name}: {len(depts)} department(s)\n\n" + "\n".join(lines))

    sources = []
    if rooms: sources.append("level1111.csv")
    if depts: sources.append("neu_hospital_departments.json")
    return {
        "answer": "\n\n---\n\n".join(blocks),
        "source": ", ".join(sources),
        "method": f"floor-listing ({floor_name}, {len(rooms)} rooms, {len(depts)} hospital depts)",
        "needs_phrasing": False,
    }

# Wrap dataset_answer so the floor check runs first. Safe to re-run this cell.
if getattr(dataset_answer, "__name__", "") != "dataset_answer_with_floors":
    _dataset_answer_base = dataset_answer

    def dataset_answer_with_floors(question: str, k: int = 30):
        hit = floor_answer(question)
        if hit is not None:
            return hit
        return _dataset_answer_base(question, k)

    dataset_answer = dataset_answer_with_floors

print("Floor-aware answers enabled. ✅")

Floor-aware answers enabled. ✅


## 12. Spell correction / normalization

In [15]:
NEU_VOCAB = [
    "hello", "hi", "hey", "greetings",
    "department", "faculty", "programme", "program", "course",
    "degree", "admission", "tuition", "scholarship", "enrol",
    "register", "bachelor", "master", "phd", "gpa",
    "where", "location", "located", "building", "floor",
    "room", "map", "campus", "directions", "find",
    "research", "hospital", "facilities", "library",
    "engineering", "dentistry", "medicine", "science",
    "arts", "business", "law", "pharmacy", "architecture",
]

# Common English words that must NEVER be auto-"corrected" into NEU_VOCAB terms.
# (Fixes bug: "there" was being matched to "where" at cutoff=0.8, turning
#  greetings like "Hi there!" into LOCATION-intent queries.)
SAFE_WORDS = {
    "there", "here", "the", "is", "are", "a", "an", "in", "on", "at",
    "what", "who", "why", "how", "when", "this", "that", "it", "to",
    "for", "of", "and", "or", "do", "does", "can", "i", "you", "me",
}

def correct_spelling(text: str) -> str:
    words = text.split()
    corrected = []
    for word in words:
        clean = word.strip("?!.,").lower()

        # Skip correction entirely for known safe/common words and very short words
        # (short words produce too many spurious fuzzy matches).
        if clean in SAFE_WORDS or len(clean) <= 3:
            corrected.append(word)
            continue

        matches = get_close_matches(clean, NEU_VOCAB, n=1, cutoff=0.85)
        if matches and matches[0] != clean:
            print(f"  [spell-check] '{word}' → '{matches[0]}'")
            corrected.append(matches[0])
        else:
            corrected.append(word)
    return " ".join(corrected)

## 13. Rule-based intent classification

In [16]:
GREETING_PATTERNS = re.compile(
    r"^\s*(hi|hello|hey|greetings|good\s*(morning|afternoon|evening)|howdy|salut|merhaba|selam|"
    r"how are you|how are u|how r u|what's up|sup|naber|nas\u0131ls\u0131n)(\s+\w+)?\W*$",
    re.IGNORECASE # ignore upper and lower case
)

# Catches bot-directed questions like "where are you", "what are you", "who are you"
BOT_DIRECTED = re.compile(
    r"^\s*(where|what|who)\s+are\s+you\W*$",
    re.IGNORECASE
)

LOCATION_KEYWORDS = re.compile(
    r"\b(where|location|located|building|floor|room|map|campus|directions?|find)\b",
    re.IGNORECASE
)

ACADEMIC_KEYWORDS = re.compile(
    r"\b(departments?|facult(?:y|ies)|programmes?|programs?|courses?|degrees?|admissions?|tuition|scholarships?|enrol(?:l(?:ed|ment)?)?|register(?:ed)?|bachelors?|masters?|phd|gpa)\b",
    re.IGNORECASE
)

def detect_intent(text: str) -> str:
    """Return one of: 'greeting' | 'bot' | 'location' | 'academic' | 'general'"""
    if GREETING_PATTERNS.match(text): # match : checks only the start of the text
        return "greeting"
    if BOT_DIRECTED.match(text):
        return "bot"
    if LOCATION_KEYWORDS.search(text): # search: checks whole of text
        return "location"
    if ACADEMIC_KEYWORDS.search(text):
        return "academic"
    return "general"

# Smoke-test
samples = [
    ("Hello!",                              "greeting"),
    ("Where are you?",                      "bot"),
    ("Where is the Faculty of Engineering?","location"),
    ("What departments does NEU have?",     "academic"),
    ("Tell me about NEU research centres.", "general"),
]
for text, expected in samples:
    pred   = detect_intent(text)
    status = "✅" if pred == expected else "❌"
    print(f"{status}  '{text}'  →  {pred}")

✅  'Hello!'  →  greeting
✅  'Where are you?'  →  bot
✅  'Where is the Faculty of Engineering?'  →  location
✅  'What departments does NEU have?'  →  academic
✅  'Tell me about NEU research centres.'  →  general


## 14. Main pipeline: dataset doublecheck first, RAG + LLM fallback second

In [17]:
def format_sources(source_docs: list) -> str:
    """Convert retrieved Documents into a clean citation block."""
    seen  = set()
    lines = []
    for doc in source_docs:
        label = (
            doc.metadata.get("faculty_name")
            or doc.metadata.get("department_name")
            or doc.metadata.get("location_name")
            or doc.metadata.get("source", "NEU Knowledge Base")
        )
        # Clean up Windows paths -> keep only the filename
        if "\\" in label or "/" in label:
            label = label.replace("\\", "/").split("/")[-1]
            label = (label.replace("neu_", "")
                         .replace("_", " ")
                         .replace(".json", "")
                         .replace(".txt", "")
                         .replace(".md", "")
                         .title())
        if label not in seen:
            seen.add(label)
            lines.append(f"  \u2022 {label}")
    return "Sources retrieved:\n" + "\n".join(lines) if lines else ""


GREETING_RESPONSE = (
    "Hello! \U0001F44B Welcome to NEU Assistant. I'm here to help you with anything about "
    "Near East University \u2014 faculties, departments, campus facilities, research, "
    "admissions, and more. What would you like to know?"
)

BOT_RESPONSE = (
    "I'm NEU Assistant, an AI chatbot for Near East University (NEU) in Nicosia, Cyprus. "
    "I'm here to help you with questions about faculties, departments, admissions, "
    "campus facilities, research, and more. How can I help you?"
)

# Note: GREETING_PATTERNS is defined once, in step 13, and already includes
# "how are you" / "what's up" etc. so small local LLMs re-greeting on every
# answer still get caught by strip_greeting_leadin below.

GREETING_LEADIN_RE = re.compile(
    r"^\s*(hello|hi|hey)[!,.]?\s*(i'?m|i am)?\s*(neu assistant)?[^.\n]*[.!\n]\s*",
    re.IGNORECASE
)

def strip_greeting_leadin(answer: str) -> str:
    """Remove a redundant greeting/self-introduction sentence the LLM
    sometimes prepends, even on non-greeting questions."""
    cleaned = GREETING_LEADIN_RE.sub("", answer, count=1).strip()
    # Guard: if stripping wiped out the whole answer, keep the original.
    return cleaned if cleaned else answer


PHRASING_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the user's question using ONLY the information in the context below. "
        "If the context describes more than one distinct matching location or item, "
        "list each one separately and clearly (e.g. as a short list), don't just pick one. "
        "Do not invent information that isn't in the context.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# Catches the LLM admitting the retrieved context didn't actually answer the
# question -- e.g. "the provided context does not contain information about...",
# "there is no information regarding...". When this fires, the dataset "match"
# wasn't a real answer, so we should fall back to the LLM's own knowledge
# instead of surfacing a dead-end reply.
_NO_ANSWER_IN_CONTEXT_RE = re.compile(
    r"\b(context (?:does not|doesn't|do not) (?:contain|provide|include|mention|state)|"
    r"no information (?:is )?(?:available|provided|found)?|"
    r"not (?:mentioned|found|available|stated|specified) in the (?:provided )?context|"
    r"based on the provided context, there is no)\b",
    re.IGNORECASE,
)

# Alias phrases that all mean "the International Building". Rewritten to the
# exact phrase that already matches the dataset, before retrieval runs.
_BUILDING_ALIASES = [
    (re.compile(
        r"\binternational\s+(?:students?\s+)?(?:affairs\s+)?"
        r"(?:office|offices|building|bldg|centre|center)\b|\biso\b",
        re.IGNORECASE),
     "international building"),
]

def _apply_aliases(text: str) -> str:
    for pattern, replacement in _BUILDING_ALIASES:
        text = pattern.sub(replacement, text)
    return text


def ask(question: str, verbose: bool = True) -> dict:
    question = _apply_aliases(correct_spelling(question))   # <-- was: correct_spelling(question)
    intent = detect_intent(question)

    if intent == "greeting":
        result = {"intent": intent, "answer": GREETING_RESPONSE, "sources": "", "method": "rule-based"}

    elif intent == "bot":
        result = {"intent": intent, "answer": BOT_RESPONSE, "sources": "", "method": "rule-based"}

    else:
        ds_hit = dataset_answer(question)
        answer_text = None

        if ds_hit is not None:
            if ds_hit.get("needs_phrasing"):
                phrased = llm.invoke(
                    PHRASING_PROMPT.format(context=ds_hit["answer"], question=question)
                )
                answer_text = strip_greeting_leadin(phrased.content)

                # NEW: chunks passed the matching filters but didn't actually answer
                # the question -- treat this the same as dataset_answer() returning
                # None, so we fall through to the general-knowledge LLM below.
                if _NO_ANSWER_IN_CONTEXT_RE.search(answer_text):
                    ds_hit = None
                    answer_text = None
            else:
                answer_text = ds_hit["answer"]

        if ds_hit is not None:
            result = {
                "intent": intent,
                "answer": answer_text,
                "sources": f"Sources retrieved:\n  • {ds_hit['source']}",
                "method": ds_hit["method"],
            }
        else:
            # --- NO MATCH (or the match didn't actually answer it): plain LLM ---
            plain = llm.invoke(GENERAL_FALLBACK_TEMPLATE.format(question=question))
            result = {
                "intent": intent,
                "answer": strip_greeting_leadin(plain.content),
                "sources": "",
                "method": "llm-fallback (general knowledge)",
            }

    if verbose:
        print(f"[{result['intent'].upper()}]  (method: {result['method']})")
        print(result["answer"])
        if result["sources"]:
            print("\n" + result["sources"])

    return result

print("ask() finalized: dataset match -> dataset (phrased if needed); no match -> plain LLM. ✅")

ask() finalized: dataset match -> dataset (phrased if needed); no match -> plain LLM. ✅


In [ ]:
import time, csv
import numpy as np

# --- count LLM calls without touching ask() ---------------------------------
class CountingLLM:
    def __init__(self, real): self._real, self.calls = real, 0
    def invoke(self, *a, **k):
        self.calls += 1
        return self._real.invoke(*a, **k)
    def __getattr__(self, n): return getattr(self._real, n)

_real_llm = llm
llm = CountingLLM(_real_llm)

# (category, question, expected_route, auto_check or None)
TESTS = [
 ("Academic","What departments are in the Faculty of Architecture?",2,"interior architecture"),
 ("Academic","Which degrees does the Faculty of Tourism offer?",2,"bsc"),
 ("Academic","What is the website of the Faculty of Artificial Intelligence and Informatics?",2,"aif.neu.edu.tr"),
 ("Academic","List all faculties at NEU",2,"20"),
 ("Academic","What departments does the Faculty of Civil and Environmental Engineering have?",2,"environmental engineering"),
 ("Academic","Does NEU offer a pharmacy programme?",2,None),
 ("Academic","What are the admission requirements for international students?",3,None),
 ("Academic","How much is the tuition fee?",3,None),
 ("Academic","What research centers does NEU have?",2,"28"),
 ("Academic","Is there a Faculty of Law?",2,None),
 ("Academic","Tell me about the NEU-IBM Research Centre",2,"supercomputer"),
 ("Academic","What degree types does the Faculty of Medicine offer?",2,"md"),
 ("Campus location","Where is the Grand Library?",3,"central"),
 ("Campus location","Where is the Faculty of Dentistry?",3,None),
 ("Campus location","Where is the post office on campus?",3,None),
 ("Campus location","Where is the supermarket?",3,None),
 ("Campus location","Where is the sports tower?",3,None),
 ("Campus location","Where can I find the International Students Office?",3,None),
 ("Campus location","Where is the Ataturk Culture and Congress Centre?",3,None),
 ("Campus location","How can I get to the Grand Library from the main gate?",3,None),
 ("Campus location","Where is the campus bookstore?",3,None),
 ("Campus location","Where is the Cyprus Car Museum?",3,None),
 ("Campus location","Where are the student dormitories?",3,None),
 ("Campus location","Where is NEU Lake?",3,None),
 ("Rooms and people","Room U08",2,"grand library"),
 ("Rooms and people","Where is room 109?",2,"innovation building"),
 ("Rooms and people","Where is Cagri Ozkan?",2,"109"),
 ("Rooms and people","Where is Necibe Can?",2,"115"),
 ("Rooms and people","Where are the bathrooms in the Innovation Building?",2,"innovation building"),
 ("Rooms and people","Where is the kitchen in the International Building?",2,"kitchen"),
 ("Rooms and people","Which rooms are on the first floor of the Innovation Building?",2,"innovation building"),
 ("Rooms and people","Where is the technical room?",2,"technical room"),
 ("Rooms and people","Computer lab in the Innovation Building",2,"computer lab"),
 ("Rooms and people","Where is Prof. Dr. Rahib Abiyev?",2,"109"),
 ("Hospital","What are the hospital departments?",2,"55"),
 ("Hospital","Hospital info",2,"209"),
 ("Hospital","Where is the emergency department?",3,"1st floor"),
 ("Hospital","What is the phone number of the hospital?",2,"444 0 535"),
 ("Hospital","How many patient rooms does the hospital have?",2,"209"),
 ("Hospital","What departments are on the third floor of the hospital?",2,"14"),
 ("Hospital","What does the Dental Hospital include?",2,"8"),
 ("Hospital","Is the emergency department open 24 hours?",3,"24/7"),
 ("Hospital","How many VIP rooms are in the hospital?",2,"22"),
 ("Hospital","Where is the pediatric cardiology department?",3,None),
 ("Misspelled","where is hopitl",2,"neu hospital"),
 ("Misspelled","emregcney medicine",3,"emergency medicine"),
 ("Misspelled","i need tourism dep",2,"tourism"),
 ("Misspelled","faculty of dentitry departments",2,"dentistry"),
 ("Misspelled","wher is the grand libary",3,None),
 ("Misspelled","bathroms in innovtion building",2,"innovation building"),
 ("Misspelled","adminsion requirments",3,None),
 ("Misspelled","wc international buliding",2,"international building"),
 ("Misspelled","reserch centres",2,"research centers"),
 ("Misspelled","faculty of enginering departments",2,None),
 ("Conversational","Hello",1,"welcome to neu assistant"),
 ("Conversational","Merhaba",1,"welcome to neu assistant"),
 ("Conversational","Who are you?",1,"neu assistant"),
 ("Out of scope","What is the capital of France?",4,None),
 ("Out of scope","What is the weather in Nicosia today?",4,None),
 ("Out of scope","Can you recommend a good restaurant near the university?",4,None),
]

def _route(method, calls):
    if method == "rule-based": return 1
    if method.startswith("llm-fallback"): return 4
    return 3 if calls >= 1 else 2

rows = []
for i, (cat, q, exp, chk) in enumerate(TESTS, 1):
    llm.calls = 0
    t0 = time.perf_counter()
    try:
        r = ask(q, verbose=False)
        ans, method, err = r["answer"], r["method"], ""
    except Exception as e:                       # quota / network errors
        ans, method, err = "", "error", str(e)[:120]
    dt = time.perf_counter() - t0
    route = _route(method, llm.calls) if not err else None
    auto = "" if chk is None else (chk.lower() in ans.lower())
    rows.append(dict(no=i, category=cat, question=q, expected_route=exp, route=route,
                     route_match=(route == exp), llm_calls=llm.calls,
                     latency_s=round(dt, 2), method=method, auto_check=auto,
                     answer=ans.replace("\n", " ")[:400], error=err, grade=""))
    print(f"{i:02d} route {route} (exp {exp}) calls={llm.calls} {dt:.2f}s  {q[:50]}")
    if llm.calls: time.sleep(2)                  # be gentle with the quota

with open("neu_eval_results.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)

# --- summary -----------------------------------------------------------------
ok = [r for r in rows if not r["error"]]
n = len(ok)
print(f"\nRan {n}/60 | errors: {60-n}")
for rt in (1, 2, 3, 4):
    rr = [r for r in ok if r["route"] == rt]
    if rr:
        lat = [r["latency_s"] for r in rr]
        print(f"Route {rt}: n={len(rr)} ({100*len(rr)/n:.1f}%)  mean={np.mean(lat):.2f}s  median={np.median(lat):.2f}s")
lat = [r["latency_s"] for r in ok]
print(f"All: mean={np.mean(lat):.2f}s median={np.median(lat):.2f}s p95={np.percentile(lat,95):.2f}s")
tot = sum(r["llm_calls"] for r in ok)
print(f"LLM calls: {tot} vs {n} baseline -> {100*(n-tot)/n:.1f}% fewer")
print("Zero-LLM-call share:", f"{100*sum(r['llm_calls']==0 for r in ok)/n:.1f}%")
print("Route matches expected:", f"{100*sum(r['route_match'] for r in ok)/n:.1f}%")
auto = [r for r in ok if r["auto_check"] != ""]
print(f"Auto-check pass: {sum(r['auto_check'] for r in auto)}/{len(auto)}")

llm = _real_llm   # restore

  [spell-check] 'departments' → 'department'
01 route 2 (exp 2) calls=0 0.01s  What departments are in the Faculty of Architectur
  [spell-check] 'degrees' → 'degree'
02 route 2 (exp 2) calls=0 0.00s  Which degrees does the Faculty of Tourism offer?
03 route 2 (exp 2) calls=0 0.00s  What is the website of the Faculty of Artificial I
04 route 2 (exp 2) calls=0 0.00s  List all faculties at NEU
  [spell-check] 'departments' → 'department'
05 route 2 (exp 2) calls=0 0.00s  What departments does the Faculty of Civil and Env
06 route 4 (exp 2) calls=1 451.80s  Does NEU offer a pharmacy programme?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 8h15m26.319392011s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.1-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value

07 route None (exp 3) calls=1 220.89s  What are the admission requirements for internatio
08 route 2 (exp 3) calls=0 0.05s  How much is the tuition fee?
09 route 2 (exp 2) calls=0 0.00s  What research centers does NEU have?
10 route 2 (exp 2) calls=0 0.00s  Is there a Faculty of Law?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite
Please retry in 8h14m20.815542967s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.1-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value

## 15. Smoke tests

In [ ]:
# Greeting
ask("Hello!")

# Academic
ask("What departments are in the Faculty of Engineering?")

# Location
ask("Where is the Grand Library on campus?")

# General
ask("What research centers does NEU have?")

[GREETING]  (method: rule-based)
Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, departments, campus facilities, research, admissions, and more. What would you like to know?
  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: faculty-match (Faculty of Civil and Environmental Engineering, score=1))
This document is about the Faculty of Civil and Environmental Engineering.

Faculty Name: Faculty of Civil and Environmental Engineering

Departments:
- Civil Engineering
- Environmental Engineering

Degree Types:
BSc, MSc, PhD

Website:
https://insaatvecevre.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json
[LOCATION]  (method: llm-fallback (general knowledge))
The Grand Library is located at the heart of the Near East University campus in Nicosia. It is a prominent, large building situated near the central square, easily accessible from most faculties and student dormitories. If you are arriving from

{'intent': 'general',
 'answer': "NEU has 28 research centers in total. The dataset provides named details for 7 of them:\n\nThis document describes a NEU research center.\n\nName: NEU-IBM Research Centre\nDescription: Houses NEU's supercomputer; conducts research across multiple disciplines\nResearch areas: Medical sciences, Chemistry, Physics, Astronomy, Architecture, Engineering, Mathematics\nNotable projects: Help Conquer Cancer project, CERN Large Hadron Collider experiment, Ebola virus research (2014), Northern Cyprus Research Area (European Research Area framework)\n\n---\n\nThis document describes a NEU research center.\n\nName: DESAM Research Institute\nWebsite: https://neu.edu.tr/research/desam-research-institute/?lang=en\n\n---\n\nThis document describes a NEU research center.\n\nName: Near East Institute\nWebsite: https://neu.edu.tr/research/near-east-institute/?lang=en\n\n---\n\nThis document describes a NEU research center.\n\nName: Lifelong Education Centre\nWebsite: htt

In [ ]:
print(llm.model)

models/gemini-3.1-flash-lite


In [ ]:
import inspect
print(llm.model)

models/gemini-3.1-flash-lite


In [ ]:
import inspect
print(inspect.getsource(type(llm).__init__)[:200])  # just to confirm object type
print("---")
print(llm.model)
print("---")
print(llm)

    def __init__(self, **kwargs: Any) -> None:
        """Needed for arg validation."""
        # Get all valid field names, including aliases
        valid_fields = set()
        for field_name, fiel
---
models/gemini-3.1-flash-lite
---
model='models/gemini-3.1-flash-lite' google_api_key=SecretStr('**********') client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000002CE178199D0> default_metadata=() model_kwargs={}


In [ ]:
test_questions = [
    "Hi there!",
    "Good morning!",
    "What departments are in the Faculty of Arts and Sciences?",
    "What degree types does the Faculty of Dentistry offer?",
    "What are the admission requirements at NEU?",
    "Where is the Grand Library?",
    "Where can I find the surgery room in the hospital?",
    "Does NEU have an Artificial Intelligence programme?",
    "What is the website of the Faculty of Dentistry?",
    "where is innovation bulding"
]

for q in test_questions:
    r = ask(q, verbose=False)
    preview = r['answer'][:120].replace("\n", " ")
    print(f"[{r['intent'].upper():<8}] {q}")
    print(f"           method \u2192 {r['method']}")
    print(f"           answer \u2192 {preview}...\n")


[GREETING] Hi there!
           method → rule-based
           answer → Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, department...

[GREETING] Good morning!
           method → rule-based
           answer → Hello! 👋 Welcome to NEU Assistant. I'm here to help you with anything about Near East University — faculties, department...

  [spell-check] 'departments' → 'department'
  [spell-check] 'Sciences?' → 'science'
[ACADEMIC] What departments are in the Faculty of Arts and Sciences?
           method → faculty-match (Faculty of Arts and Sciences, score=1)
           answer → This document is about the Faculty of Arts and Sciences.  Faculty Name: Faculty of Arts and Sciences  Departments: - Dep...

[ACADEMIC] What degree types does the Faculty of Dentistry offer?
           method → faculty-match (Faculty of Dentistry, score=1)
           answer → This document is about the Faculty of Dentistry.  Faculty Name: Faculty of De

[ACADEMIC] What are the admission requirements at NEU?
           method → llm-fallback (general knowledge)
           answer → Admission requirements for Near East University (NEU) generally depend on the level of study you are applying for (Under...

[LOCATION] Where is the Grand Library?
           method → dataset-match (multiple: 29 results)
           answer → The Grand Library is located in the following places:  *   **Main Campus:** Central area *   **Campus Layout:** A large,...



In [ ]:
ask("Where is the Grand Library on campus?")

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite
Please retry in 28.432734408s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.1-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1

[LOCATION]  (method: llm-fallback (general knowledge))
The Grand Library at Near East University is located at the heart of the main campus in Nicosia. It is a prominent landmark situated near the Atatürk Culture and Congress Center and is easily accessible from the main university entrance. You will find it centrally located among the faculty buildings; if you are using a campus map, look for the large, iconic building complex that serves as the primary hub for research and study.


{'intent': 'location',
 'answer': 'The Grand Library at Near East University is located at the heart of the main campus in Nicosia. It is a prominent landmark situated near the Atatürk Culture and Congress Center and is easily accessible from the main university entrance. You will find it centrally located among the faculty buildings; if you are using a campus map, look for the large, iconic building complex that serves as the primary hub for research and study.',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("Room U08")

[LOCATION]  (method: exact-room-match (U08))
Room U08 ("Grand Library Archives") is located in the Innovation Building, level_u (underground floor).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'Room U08 ("Grand Library Archives") is located in the Innovation Building, level_u (underground floor).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'exact-room-match (U08)'}

In [ ]:
ask("Çagri Özkan, MSe")

[GENERAL]  (method: exact-lecturer-match (Çagri Özkan, MSe))
Çagri Özkan, MSe (Information Systems Engineering) is based in Room 109 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr. Rahib Abiyev (Computer Engineering - HoD), Assoc. Prof. Dr.John Bush (Cyber Security Engineering - HoD), Asst. Prof. Dr. Amr Abdelbari (Information Systems
Engineering-HoD).

Sources retrieved:
  • level1111.csv


{'intent': 'general',
 'answer': 'Çagri Özkan, MSe (Information Systems Engineering) is based in Room 109 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr. Rahib Abiyev (Computer Engineering - HoD), Assoc. Prof. Dr.John Bush (Cyber Security Engineering - HoD), Asst. Prof. Dr. Amr Abdelbari (Information Systems\nEngineering-HoD).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'exact-lecturer-match (Çagri Özkan, MSe)'}

## 16. Flask backend

In [ ]:
# ============================================================
# PASTE THIS AS A NEW CELL AT THE END OF YOUR NOTEBOOK
# (must run AFTER the cells that define: splitter, vectorstore,
#  Document, and ask() — i.e. after cell 20 / the RAG chain setup)
#
# Starts a Flask web server so the HTML chatbot UI *and* the new
# admin page can:
#   - send questions to your ask() function
#   - add new documents straight into your FAISS vectorstore
#   - list what's currently indexed
#   - delete a document's chunks from the vectorstore
#   - update (replace) a document's chunks in place
# ============================================================

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import os
import uuid
from datetime import datetime

app = Flask(__name__)
CORS(app)  # allows your frontend to call it from a different port

# ------------------------------------------------------------
# Where uploaded files get saved on disk, and a running record of
# what's been added to the vectorstore this session.
#
# Each item now also stores the exact vectorstore chunk `ids` that
# came back from `vectorstore.add_documents(...)`, so we can later
# delete or replace *just* that document's chunks without touching
# anything else in the index.
# ------------------------------------------------------------
UPLOAD_DIR = "uploaded_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)

ALLOWED_EXTENSIONS = {".txt", ".pdf", ".docx", ".csv", ".md"}

# Each item: {"id", "filename", "chunks_added", "ids": [...], "uploaded_at"}
uploaded_documents = []


def extract_text(filepath, ext):
    """Pull plain text out of the uploaded file based on its extension."""
    if ext in (".txt", ".md", ".csv"):
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    if ext == ".pdf":
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader  # fallback if pypdf isn't installed
        reader = PdfReader(filepath)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    if ext == ".docx":
        import docx
        doc = docx.Document(filepath)
        return "\n".join(p.text for p in doc.paragraphs)

    return ""


def find_document(doc_id):
    return next((d for d in uploaded_documents if d["id"] == doc_id), None)


def add_document_to_index(text, filename):
    """
    Wraps the uploaded text in a LangChain Document, splits it with the
    SAME splitter used to build the original knowledge base, and adds
    the resulting chunks directly into the existing `vectorstore`.

    Because `retriever` and `dataset_answer()` both read from this same
    `vectorstore` object (not a copy), new content becomes searchable by
    both the RAG/LLM path and the no-LLM doublecheck path immediately —
    no restart, no rebuild.

    Returns (chunk_ids, num_chunks) so the caller can track what was
    added and delete/replace it later.
    """
    new_doc = Document(
        page_content=text,
        metadata={"source": filename, "type": "uploaded"},
    )
    new_chunks = splitter.split_documents([new_doc])
    chunk_ids = vectorstore.add_documents(new_chunks)

    print(f"📄 Added '{filename}' to vectorstore as {len(new_chunks)} chunk(s).")
    return chunk_ids, len(new_chunks)


def remove_chunks_from_index(chunk_ids):
    """Delete a specific set of chunk ids from the vectorstore, if any exist."""
    if chunk_ids:
        vectorstore.delete(ids=chunk_ids)


@app.route("/ask", methods=["POST"])
def ask_endpoint():
    data = request.get_json()
    question = data.get("question", "").strip()
    if not question:
        return jsonify({"error": "No question provided"}), 400
    try:
        result = ask(question, verbose=False)
        return jsonify({
            "answer": result["answer"],
            "intent": result["intent"],
            "sources": result["sources"]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/upload", methods=["POST"])
def upload_endpoint():
    if "file" not in request.files:
        return jsonify({"error": "No file included in the request."}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400

    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        return jsonify({
            "error": f"Unsupported file type '{ext}'. "
                     f"Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}"
        }), 400

    save_path = os.path.join(UPLOAD_DIR, file.filename)
    file.save(save_path)

    try:
        text = extract_text(save_path, ext)
    except Exception as e:
        return jsonify({"error": f"Could not read file: {e}"}), 500

    if not text.strip():
        return jsonify({"error": "No readable text found in file."}), 400

    try:
        chunk_ids, chunks_added = add_document_to_index(text, file.filename)
    except Exception as e:
        return jsonify({"error": f"Could not index file: {e}"}), 500

    entry = {
        "id": str(uuid.uuid4()),
        "filename": file.filename,
        "chunks_added": chunks_added,
        "ids": chunk_ids,
        "uploaded_at": datetime.utcnow().isoformat() + "Z",
    }
    uploaded_documents.append(entry)

    return jsonify({
        "message": f"'{file.filename}' uploaded and added to the knowledge base.",
        "document": {k: v for k, v in entry.items() if k != "ids"},
    })


@app.route("/documents", methods=["GET"])
def list_documents():
    """See what's been uploaded (and indexed) this session."""
    # Don't leak raw vectorstore chunk ids to the frontend — not needed there.
    docs = [{k: v for k, v in d.items() if k != "ids"} for d in uploaded_documents]
    return jsonify({"documents": docs})


@app.route("/documents/<doc_id>", methods=["DELETE"])
def delete_document(doc_id):
    """Remove a document's chunks from the vectorstore and drop its file."""
    entry = find_document(doc_id)
    if not entry:
        return jsonify({"error": "Document not found."}), 404

    try:
        remove_chunks_from_index(entry["ids"])
    except Exception as e:
        return jsonify({"error": f"Could not remove from vectorstore: {e}"}), 500

    save_path = os.path.join(UPLOAD_DIR, entry["filename"])
    if os.path.exists(save_path):
        os.remove(save_path)

    uploaded_documents.remove(entry)
    return jsonify({"message": f"'{entry['filename']}' removed from the knowledge base."})


@app.route("/documents/<doc_id>", methods=["PUT"])
def update_document(doc_id):
    """
    Replace a document with a new version: delete its old chunks from the
    vectorstore, then index the newly uploaded file in its place.
    """
    entry = find_document(doc_id)
    if not entry:
        return jsonify({"error": "Document not found."}), 404

    if "file" not in request.files:
        return jsonify({"error": "No replacement file included in the request."}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400

    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        return jsonify({
            "error": f"Unsupported file type '{ext}'. "
                     f"Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}"
        }), 400

    save_path = os.path.join(UPLOAD_DIR, file.filename)
    file.save(save_path)

    try:
        text = extract_text(save_path, ext)
    except Exception as e:
        return jsonify({"error": f"Could not read file: {e}"}), 500

    if not text.strip():
        return jsonify({"error": "No readable text found in file."}), 400

    try:
        new_ids, new_count = add_document_to_index(text, file.filename)
        remove_chunks_from_index(entry["ids"])  # remove old only after new succeeds
    except Exception as e:
        return jsonify({"error": f"Could not index replacement file: {e}"}), 500

    # Clean up old saved file if the filename changed
    old_path = os.path.join(UPLOAD_DIR, entry["filename"])
    if entry["filename"] != file.filename and os.path.exists(old_path):
        os.remove(old_path)

    entry["filename"] = file.filename
    entry["chunks_added"] = new_count
    entry["ids"] = new_ids
    entry["uploaded_at"] = datetime.utcnow().isoformat() + "Z"

    return jsonify({
        "message": f"'{file.filename}' replaced the previous version in the knowledge base.",
        "document": {k: v for k, v in entry.items() if k != "ids"},
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

# Run in a background thread so Jupyter doesn't freeze
thread = threading.Thread(target=lambda: app.run(port=5050, debug=False, use_reloader=False))
thread.daemon = True
thread.start()
print("Flask server running on http://localhost:5050 ✅")
print("Chat UI hits /ask. Admin page hits /documents (GET/DELETE/PUT) and /upload (POST).")


Flask server running on http://localhost:5050 ✅
Chat UI hits /ask. Admin page hits /documents (GET/DELETE/PUT) and /upload (POST).


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit
127.0.0.1 - - [14/Oct/2026 17:56:41] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 17:56:41] "POST /ask HTTP/1.1" 200 -


  [spell-check] 'Departments' → 'department'


127.0.0.1 - - [14/Oct/2026 17:56:54] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 17:57:02] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 17:57:26] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 17:57:26] "POST /ask HTTP/1.1" 200 -


...

  [spell-check] 'locate' → 'located'


127.0.0.1 - - [14/Oct/2026 18:46:46] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 18:47:09] "OPTIONS /ask HTTP/1.1" 200 -


  [spell-check] 'locate' → 'located'


127.0.0.1 - - [14/Oct/2026 18:47:27] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 18:47:30] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 18:47:35] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 18:47:59] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 18:48:00] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:20:35] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:21:12] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:22:44] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:22:45] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:23:22] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:23:23] "POST /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:24:21] "OPTIONS /ask HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 19:24:21] "POST /ask HTTP/1.1" 200 -


  [spell-check] 'departments' → 'department'


In [ ]:
ask("where is international office")
ask("where is international building")

[LOCATION]  (method: dataset-match (chunk, score=0.55))
The International Student Office (ISO) building is located on the Main Campus.

Sources retrieved:
  • neu_campus_facilities.json
[LOCATION]  (method: dataset-match (chunk, score=0.55))
The International Student Office (ISO) is located in the Main Campus building.

Sources retrieved:
  • neu_campus_facilities.json


{'intent': 'location',
 'answer': 'The International Student Office (ISO) is located in the Main Campus building.',
 'sources': 'Sources retrieved:\n  • neu_campus_facilities.json',
 'method': 'dataset-match (chunk, score=0.55)'}

In [ ]:
ask("where is grand library")

[LOCATION]  (method: dataset-match (multiple: 30 results))
The Grand Library is located in the central area of the main campus. It is a large, standalone building that is signposted from all main campus roads.

Sources retrieved:
  • neu_phone_directory.json, neu_qa_dataset.txt, neu_campus_facilities.json, Grand Library (Near East University Grand Library), neu_admissions_academic.json, Book House (Campus Bookstore)


{'intent': 'location',
 'answer': 'The Grand Library is located in the central area of the main campus. It is a large, standalone building that is signposted from all main campus roads.',
 'sources': 'Sources retrieved:\n  • neu_phone_directory.json, neu_qa_dataset.txt, neu_campus_facilities.json, Grand Library (Near East University Grand Library), neu_admissions_academic.json, Book House (Campus Bookstore)',
 'method': 'dataset-match (multiple: 30 results)'}

In [ ]:
ask("where is mosque at neu")

[LOCATION]  (method: llm-fallback (general knowledge))
The Near East University Mosque, known as the Müzeyyen İbrahimoğlu Mosque, is located on the main NEU campus in Nicosia. It is situated prominently near the main entrance area, close to the Faculty of Dentistry and the Grand Library. You can easily spot it by its distinctive architecture and minarets as you enter the campus grounds.


{'intent': 'location',
 'answer': 'The Near East University Mosque, known as the Müzeyyen İbrahimoğlu Mosque, is located on the main NEU campus in Nicosia. It is situated prominently near the main entrance area, close to the Faculty of Dentistry and the Grand Library. You can easily spot it by its distinctive architecture and minarets as you enter the campus grounds.',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("where is neu")

[LOCATION]  (method: llm-fallback (general knowledge))
Near East University (NEU) is located in Nicosia, the capital city of Northern Cyprus. The main campus is situated in the Near East Boulevard area, just a short drive from the city center. 

If you are planning to visit, you can easily reach the campus by local buses, taxis, or private transport. Would you like me to help you find specific directions or information on how to get there from a certain part of the city?


{'intent': 'location',
 'answer': 'Near East University (NEU) is located in Nicosia, the capital city of Northern Cyprus. The main campus is situated in the Near East Boulevard area, just a short drive from the city center. \n\nIf you are planning to visit, you can easily reach the campus by local buses, taxis, or private transport. Would you like me to help you find specific directions or information on how to get there from a certain part of the city?',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("where is grand library")

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite
Please retry in 36.84179633s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.1-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15

[LOCATION]  (method: dataset-match (multiple: 30 results))
The Grand Library is located in the central area of the main campus. It is a large, standalone building that is signposted from all main campus roads.

Sources retrieved:
  • neu_phone_directory.json, neu_qa_dataset.txt, neu_campus_facilities.json, Grand Library (Near East University Grand Library), neu_admissions_academic.json, Book House (Campus Bookstore)


{'intent': 'location',
 'answer': 'The Grand Library is located in the central area of the main campus. It is a large, standalone building that is signposted from all main campus roads.',
 'sources': 'Sources retrieved:\n  • neu_phone_directory.json, neu_qa_dataset.txt, neu_campus_facilities.json, Grand Library (Near East University Grand Library), neu_admissions_academic.json, Book House (Campus Bookstore)',
 'method': 'dataset-match (multiple: 30 results)'}

In [ ]:
ask("where is kitchen at international building")

[LOCATION]  (method: dataset-match (multiple: 3 results, itemized))
There is more than one match:

- Room Kitchen ("Kitchen"), located in the International Building, Second Floor.
- Room Kitchen ("Kitchen"), located in the International Building, First Floor.
- Room Kitchen ("Kitchen"), located in the International Building, ground floor.

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room Kitchen ("Kitchen"), located in the International Building, Second Floor.\n- Room Kitchen ("Kitchen"), located in the International Building, First Floor.\n- Room Kitchen ("Kitchen"), located in the International Building, ground floor.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 3 results, itemized)'}

In [ ]:
ask("how to go to neu")

[GENERAL]  (method: llm-fallback (general knowledge))
To get to Near East University (NEU) in Nicosia, Northern Cyprus, you have a few convenient options:

*   **By Car or Taxi:** The university is located in the Near East Boulevard (Yakın Doğu Bulvarı) area. Most taxi drivers in Nicosia are very familiar with the campus; simply ask for "Yakın Doğu Üniversitesi."
*   **Public Transportation:** There are various shuttle services and public buses that run regularly from the city center of Nicosia (Lefkoşa) to the NEU campus. You can check the local bus schedules at the main bus terminal.
*   **From Ercan International Airport (ECN):** The airport is approximately 20–25 minutes away by car. You can take an airport shuttle (KIBHAS) to the city center and then take a taxi or bus to the campus, or arrange a private airport transfer directly to the university.

If you are already on campus and need to find a specific building, please let me know which one, and I would be happy to help you loc

{'intent': 'general',
 'answer': 'To get to Near East University (NEU) in Nicosia, Northern Cyprus, you have a few convenient options:\n\n*   **By Car or Taxi:** The university is located in the Near East Boulevard (Yakın Doğu Bulvarı) area. Most taxi drivers in Nicosia are very familiar with the campus; simply ask for "Yakın Doğu Üniversitesi."\n*   **Public Transportation:** There are various shuttle services and public buses that run regularly from the city center of Nicosia (Lefkoşa) to the NEU campus. You can check the local bus schedules at the main bus terminal.\n*   **From Ercan International Airport (ECN):** The airport is approximately 20–25 minutes away by car. You can take an airport shuttle (KIBHAS) to the city center and then take a taxi or bus to the campus, or arrange a private airport transfer directly to the university.\n\nIf you are already on campus and need to find a specific building, please let me know which one, and I would be happy to help you locate it!',
 'so

In [ ]:
ask("where is grand library at innovation building")

[LOCATION]  (method: dataset-match (chunk, score=0.57))
The Grand Library Archives is located in the Innovation building on level_u (underground floor) in Room U08.

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'The Grand Library Archives is located in the Innovation building on level_u (underground floor) in Room U08.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (chunk, score=0.57)'}

In [ ]:
ask("where is grand library at innovation building")

[LOCATION]  (method: dataset-match (chunk, score=0.57))
The Grand Library Archives is located in the innovation building on level_u (underground floor), Room U08.

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'The Grand Library Archives is located in the innovation building on level_u (underground floor), Room U08.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (chunk, score=0.57)'}

In [ ]:
ask("where is technical room")

[LOCATION]  (method: dataset-match (multiple: 6 results, itemized))
There is more than one match:

- RoomTECHNICAL ROOM ("TECHNICAL ROOM"), located in the International Building, Second Floor.
- Room Technical Room, located in the International Building, First Floor.
- Room Technical Room ("Technical Room"), located in the International Building, ground floor.
- Room 118 ("Technical Room 118"), located in the Innovation Building, level 1 (first floor).
- Room U09 ("Technical Room"), located in the Innovation Building, level_u (underground floor).
- Room 214 ("Technical Room 214"), located in the Innovation Building, level 2 (second floor).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- RoomTECHNICAL ROOM ("TECHNICAL ROOM"), located in the International Building, Second Floor.\n- Room Technical Room, located in the International Building, First Floor.\n- Room Technical Room ("Technical Room"), located in the International Building, ground floor.\n- Room 118 ("Technical Room 118"), located in the Innovation Building, level 1 (first floor).\n- Room U09 ("Technical Room"), located in the Innovation Building, level_u (underground floor).\n- Room 214 ("Technical Room 214"), located in the Innovation Building, level 2 (second floor).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 6 results, itemized)'}

In [ ]:
ask("where is hopitl")

  [spell-check] 'hopitl' → 'hospital'
[LOCATION]  (method: full-listing (hospital_info))
This document describes general information about NEU Hospital.
Total area: 55000 sqm
Blocks: East Block, West Block
Floors (East): Ground (0) through 4th floor
Floors (West): Ground (0) through 9th floor
Total patient rooms: 209
VIP rooms: 22
Max bed capacity: 500
Phone: +90 392 444 0 535
Website: https://neareasthospital.com
Address: Near East University Hospital, Near East Boulevard, Nicosia, North Cyprus
Appointment booking website: https://neareasthospital.com
Appointment booking phone: +90 392 444 0 535

Sources retrieved:
  • neu_hospital_departments.json


{'intent': 'location',
 'answer': 'This document describes general information about NEU Hospital.\nTotal area: 55000 sqm\nBlocks: East Block, West Block\nFloors (East): Ground (0) through 4th floor\nFloors (West): Ground (0) through 9th floor\nTotal patient rooms: 209\nVIP rooms: 22\nMax bed capacity: 500\nPhone: +90 392 444 0 535\nWebsite: https://neareasthospital.com\nAddress: Near East University Hospital, Near East Boulevard, Nicosia, North Cyprus\nAppointment booking website: https://neareasthospital.com\nAppointment booking phone: +90 392 444 0 535',
 'sources': 'Sources retrieved:\n  • neu_hospital_departments.json',
 'method': 'full-listing (hospital_info)'}

In [ ]:
ask("how to go to innovation building")

[LOCATION]  (method: llm-fallback (general knowledge))
To provide you with the most accurate directions to the Innovation Building at Near East University, could you please clarify if you are already on campus? 

Generally, the Innovation and Information Technologies Centre is located centrally on our campus in Nicosia. If you are arriving by car or university shuttle, you can follow the main campus signage towards the Faculty of Engineering area; the Innovation building is situated in close proximity to the main library and the engineering blocks.

If you are currently on campus and need specific navigation assistance, I recommend visiting the nearest security kiosk or the information desk at the Grand Library, where staff can provide you with a printed campus map and precise walking directions.


{'intent': 'location',
 'answer': 'To provide you with the most accurate directions to the Innovation Building at Near East University, could you please clarify if you are already on campus? \n\nGenerally, the Innovation and Information Technologies Centre is located centrally on our campus in Nicosia. If you are arriving by car or university shuttle, you can follow the main campus signage towards the Faculty of Engineering area; the Innovation building is situated in close proximity to the main library and the engineering blocks.\n\nIf you are currently on campus and need specific navigation assistance, I recommend visiting the nearest security kiosk or the information desk at the Grand Library, where staff can provide you with a printed campus map and precise walking directions.',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("how to go to innovation building")

[LOCATION]  (method: llm-fallback (general knowledge))
To provide you with the most accurate directions to the Innovation Building at Near East University, could you please clarify which specific "Innovation Building" you are referring to? 

Our campus hosts several research and technology centers, such as the **Innovation and Information Technologies Center** or specific labs within the **Technopark**. If you can provide a bit more detail on the department or the event you are attending, I would be happy to help you locate it on our campus map!


{'intent': 'location',
 'answer': 'To provide you with the most accurate directions to the Innovation Building at Near East University, could you please clarify which specific "Innovation Building" you are referring to? \n\nOur campus hosts several research and technology centers, such as the **Innovation and Information Technologies Center** or specific labs within the **Technopark**. If you can provide a bit more detail on the department or the event you are attending, I would be happy to help you locate it on our campus map!',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
print(_ALL_BUILDING_TOKENS)

{'building', 'international', 'innovation'}


In [ ]:
ask("where is kitchen at international building")

[LOCATION]  (method: dataset-match (multiple: 3 results, itemized))
There is more than one match:

- Room Kitchen ("Kitchen"), located in the International Building, Second Floor.
- Room Kitchen ("Kitchen"), located in the International Building, First Floor.
- Room Kitchen ("Kitchen"), located in the International Building, ground floor.

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room Kitchen ("Kitchen"), located in the International Building, Second Floor.\n- Room Kitchen ("Kitchen"), located in the International Building, First Floor.\n- Room Kitchen ("Kitchen"), located in the International Building, ground floor.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 3 results, itemized)'}

In [ ]:
print([ (d.metadata.get("room_number"), d.metadata.get("room_name"), d.metadata.get("room_type"))
        for d in ROOM_DOCS if d.metadata.get("building","").strip().lower() == "innovation building" ])

[('Room 10', 'Office', 'Office'), ('Room 101', 'Biyomedikal Lab\nBiomaterials & Biosensing\nTechnologies Lab.', 'Laboratory'), ('Room 102', 'Classroom 102', 'Classroom'), ('Room 103', 'Computer Lab 103', 'Computer Laboratory'), ('Room 104', 'Computer Lab 104', 'Computer Laboratory'), ('Room 105', 'Classroom 105', 'Classroom'), ('Room 106', 'Classroom 106', 'Classroom'), ('Room 107', 'Office', 'Office'), ('Room 108', 'Office', 'Office'), ('Room 109', 'Office', 'Office'), ('Room 11', 'Office', 'Office'), ('Room 110', 'WC 110', 'Restroom (WC) for MEN'), ('Room 111', 'WC 111', 'Restroom (WC) for WOMEN'), ('Room 112', 'Office', 'Office'), ('Room 113', 'Office', 'Office'), ('Room 114', 'Physics Lab 114', 'Physics Laboratory'), ('Room 115', 'Office', 'Office'), ('Room 117', 'Print Room 117', 'Print Room'), ('Room 118', 'Technical Room 118', 'Technical Room'), ('Room 119', 'Computer Lab 119', 'Computer Laboratory'), ('Room 120', 'Office', 'Office'), ('Room 125', 'Medical Instrumentation Showro

In [ ]:
print(_expand_synonyms({"bathroom"}))

{'restroom', 'bathroom', 'lavatory', 'toilet', 'washroom'}


In [ ]:
ask("What are the hospital departments?")

  [spell-check] 'departments?' → 'department'
[ACADEMIC]  (method: full-listing (all 55 hospital departments))
NEU Hospital has 55 departments:

- Emergency Medicine
- Allergy and Immunology
- Anesthesiology and Reanimation
- Nutrition and Dietetics
- Brain and Neurosurgery
- Biochemistry Laboratory
- Pediatric Surgery
- Pediatric Endocrinology
- Pediatric Gastroenterology
- Pediatric Immunology and Allergy
- Pediatric Cardiology
- Pediatric Nephrology
- Pediatric Neurology
- Child and Adolescent Psychiatry
- Pediatric Rheumatology
- Pediatric Health and Diseases (Pediatrics)
- Dermatology
- Endocrinology and Metabolic Diseases
- Infectious Diseases and Clinical Microbiology
- Physical Medicine and Rehabilitation
- Gastroenterology
- General Surgery
- Geriatrics
- Genetics – Medical Genetics Diagnostic Laboratory
- Interventional Radiology
- Thoracic Surgery
- Chest Diseases (Pulmonology)
- Ophthalmology (Eye Diseases)
- Haematology
- Internal Medicine
- Cardiovascular Surgery
- Gynaec

{'intent': 'academic',
 'answer': 'NEU Hospital has 55 departments:\n\n- Emergency Medicine\n- Allergy and Immunology\n- Anesthesiology and Reanimation\n- Nutrition and Dietetics\n- Brain and Neurosurgery\n- Biochemistry Laboratory\n- Pediatric Surgery\n- Pediatric Endocrinology\n- Pediatric Gastroenterology\n- Pediatric Immunology and Allergy\n- Pediatric Cardiology\n- Pediatric Nephrology\n- Pediatric Neurology\n- Child and Adolescent Psychiatry\n- Pediatric Rheumatology\n- Pediatric Health and Diseases (Pediatrics)\n- Dermatology\n- Endocrinology and Metabolic Diseases\n- Infectious Diseases and Clinical Microbiology\n- Physical Medicine and Rehabilitation\n- Gastroenterology\n- General Surgery\n- Geriatrics\n- Genetics – Medical Genetics Diagnostic Laboratory\n- Interventional Radiology\n- Thoracic Surgery\n- Chest Diseases (Pulmonology)\n- Ophthalmology (Eye Diseases)\n- Haematology\n- Internal Medicine\n- Cardiovascular Surgery\n- Gynaecology and Obstetrics\n- Female Urology\n- B

In [ ]:
ask("What does Oral and Dental Health include?")

[GENERAL]  (method: dataset-match (chunk, score=0.56))
The Dental Hospital is a separate building on the NEU campus (not in the main hospital block). It has 8 departments: Dental and Maxillofacial Radiology, Periodontology, Restorative Dentistry, Endodontics, Oral and Maxillofacial Surgery, Prosthodontics, Paedodontics, and Orthodontics. Open Monday–Friday 08:00–17:00.

Sources retrieved:
  • neu_qa_dataset.txt


{'intent': 'general',
 'answer': 'The Dental Hospital is a separate building on the NEU campus (not in the main hospital block). It has 8 departments: Dental and Maxillofacial Radiology, Periodontology, Restorative Dentistry, Endodontics, Oral and Maxillofacial Surgery, Prosthodontics, Paedodontics, and Orthodontics. Open Monday–Friday 08:00–17:00.',
 'sources': 'Sources retrieved:\n  • neu_qa_dataset.txt',
 'method': 'dataset-match (chunk, score=0.56)'}

In [ ]:
ask("third floor departments for hospital")

  [spell-check] 'departments' → 'department'
[LOCATION]  (method: floor-listing (Third floor, 0 rooms, 14 hospital depts))
NEU Hospital - Third floor: 14 department(s)

- Allergy and Immunology (East Block)
- Brain and Neurosurgery (East Block)
- Pediatric Surgery (East Block)
- Pediatric Endocrinology (East Block)
- Pediatric Gastroenterology (East Block)
- Pediatric Immunology and Allergy (East Block)
- Pediatric Cardiology (East Block)
- Pediatric Nephrology (East Block)
- Pediatric Neurology (East Block)
- Pediatric Rheumatology (East Block)
- Pediatric Health and Diseases (Pediatrics) (East Block)
- Gynaecology and Obstetrics (East Block)
- Neurology (East Block)
- Orthopaedics and Traumatology (East Block)

Sources retrieved:
  • neu_hospital_departments.json


{'intent': 'location',
 'answer': 'NEU Hospital - Third floor: 14 department(s)\n\n- Allergy and Immunology (East Block)\n- Brain and Neurosurgery (East Block)\n- Pediatric Surgery (East Block)\n- Pediatric Endocrinology (East Block)\n- Pediatric Gastroenterology (East Block)\n- Pediatric Immunology and Allergy (East Block)\n- Pediatric Cardiology (East Block)\n- Pediatric Nephrology (East Block)\n- Pediatric Neurology (East Block)\n- Pediatric Rheumatology (East Block)\n- Pediatric Health and Diseases (Pediatrics) (East Block)\n- Gynaecology and Obstetrics (East Block)\n- Neurology (East Block)\n- Orthopaedics and Traumatology (East Block)',
 'sources': 'Sources retrieved:\n  • neu_hospital_departments.json',
 'method': 'floor-listing (Third floor, 0 rooms, 14 hospital depts)'}

In [ ]:
ask("Faculties & Departments")

  [spell-check] 'Departments' → 'department'
[ACADEMIC]  (method: full-listing (all 20 faculties))
NEU has 20 faculties:

This document is about the Faculty of Architecture.

Faculty Name: Faculty of Architecture

Departments:
- Architecture (Turkish Program)
- Architecture (English Program)
- Interior Architecture

Degree Types:
BSc, MSc, PhD

Website:
https://mimarlik.neu.edu.tr

---

This document is about the Atatürk Faculty of Education.

Faculty Name: Atatürk Faculty of Education

Departments:
- Department of Computer Education and Instructional Technologies
- Philosophy Group Teaching Department
- Science Education Department
- Department of Arts and Craft Education
- Music Teaching Department
- Preschool Education Department
- Special Education Teaching Department
- Guidance and Psychological Counselling Department
- Classroom Teaching Department
- Social Studies Teaching Department

Degree Types:
BA, BSc, MA, PhD

Website:
https://egitim.neu.edu.tr

---

This document is about

{'intent': 'academic',
 'answer': 'NEU has 20 faculties:\n\nThis document is about the Faculty of Architecture.\n\nFaculty Name: Faculty of Architecture\n\nDepartments:\n- Architecture (Turkish Program)\n- Architecture (English Program)\n- Interior Architecture\n\nDegree Types:\nBSc, MSc, PhD\n\nWebsite:\nhttps://mimarlik.neu.edu.tr\n\n---\n\nThis document is about the Atatürk Faculty of Education.\n\nFaculty Name: Atatürk Faculty of Education\n\nDepartments:\n- Department of Computer Education and Instructional Technologies\n- Philosophy Group Teaching Department\n- Science Education Department\n- Department of Arts and Craft Education\n- Music Teaching Department\n- Preschool Education Department\n- Special Education Teaching Department\n- Guidance and Psychological Counselling Department\n- Classroom Teaching Department\n- Social Studies Teaching Department\n\nDegree Types:\nBA, BSc, MA, PhD\n\nWebsite:\nhttps://egitim.neu.edu.tr\n\n---\n\nThis document is about the Faculty of Denti

In [ ]:
result = dataset_answer("Research Centers")
print(result)

{'answer': "NEU has 28 research centers in total. The dataset provides named details for 7 of them:\n\nThis document describes a NEU research center.\n\nName: NEU-IBM Research Centre\nDescription: Houses NEU's supercomputer; conducts research across multiple disciplines\nResearch areas: Medical sciences, Chemistry, Physics, Astronomy, Architecture, Engineering, Mathematics\nNotable projects: Help Conquer Cancer project, CERN Large Hadron Collider experiment, Ebola virus research (2014), Northern Cyprus Research Area (European Research Area framework)\n\n---\n\nThis document describes a NEU research center.\n\nName: DESAM Research Institute\nWebsite: https://neu.edu.tr/research/desam-research-institute/?lang=en\n\n---\n\nThis document describes a NEU research center.\n\nName: Near East Institute\nWebsite: https://neu.edu.tr/research/near-east-institute/?lang=en\n\n---\n\nThis document describes a NEU research center.\n\nName: Lifelong Education Centre\nWebsite: https://neu.edu.tr/resear

In [ ]:
import inspect
print(inspect.getsource(dataset_answer))

    def dataset_answer_with_floors(question: str, k: int = 30):
        hit = floor_answer(question)
        if hit is not None:
            return hit
        return _dataset_answer_base(question, k)



In [ ]:
ask("i need tourism departments")

  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: faculty-match (Faculty of Tourism, score=1))
This document is about the Faculty of Tourism.

Faculty Name: Faculty of Tourism

Departments:
- Tourism and Hotel Management (Turkish Program)
- Tourism and Hotel Management (English Program)
- Gastronomy
- Cookery

Degree Types:
BSc, MSc

Website:
https://turizm.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'academic',
 'answer': 'This document is about the Faculty of Tourism.\n\nFaculty Name: Faculty of Tourism\n\nDepartments:\n- Tourism and Hotel Management (Turkish Program)\n- Tourism and Hotel Management (English Program)\n- Gastronomy\n- Cookery\n\nDegree Types:\nBSc, MSc\n\nWebsite:\nhttps://turizm.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Tourism, score=1)'}

In [ ]:
ask("i need ai departments")

  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: faculty-match (Faculty of Artificial Intelligence and Informatics, score=2))
This document is about the Faculty of Artificial Intelligence and Informatics.

Faculty Name: Faculty of Artificial Intelligence and Informatics

Departments:
- Artificial Intelligence
- Informatics

Degree Types:
BSc, MSc, PhD

Website:
https://aif.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'academic',
 'answer': 'This document is about the Faculty of Artificial Intelligence and Informatics.\n\nFaculty Name: Faculty of Artificial Intelligence and Informatics\n\nDepartments:\n- Artificial Intelligence\n- Informatics\n\nDegree Types:\nBSc, MSc, PhD\n\nWebsite:\nhttps://aif.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Artificial Intelligence and Informatics, score=2)'}

In [ ]:
for doc in ROOM_DOCS:
    m = doc.metadata
    if m.get("building", "").strip().lower() == "international building":
        room_type = m.get("room_type", "")
        room_name = m.get("room_name", "")
        if "bath" in room_type.lower() or "bath" in room_name.lower() or "wc" in room_type.lower() or "restroom" in room_type.lower():
            print(m.get("floor"), "|", room_name, "|", room_type)

First Floor | Gents Room | Restroom
First Floor | Ladies Room | Restroom
Underground | Men Bathroom | Restroom
Second Floor | Men's Restroom | Restroom
Underground | Special Needs Bathroom | Restroom
ground floor | Staff Restroom (WC) | Restroom
ground floor | WC | Restroom (WC) for WOMEN
Underground | Women Bathroom | Restroom
Second Floor | Women's Restroom | Restroom
ground floor | WC | Restroom (WC) for WOMEN


In [ ]:
ask("bathrooms at innovation building")

[LOCATION]  (method: dataset-match (multiple: 7 results, itemized))
There is more than one match:

- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).
- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).
- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).
- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).
- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).
- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).
- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).\n- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).\n- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).\n- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).\n- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).\n- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).\n- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 7 results, itemized)'}

In [ ]:
ask("bathrooms at international building")

[LOCATION]  (method: dataset-match (multiple: 10 results, itemized))
There is more than one match:

- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.
- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.
- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.
- Room Staff Restroom (WC) ("Staff Restroom (WC)"), located in the International Building, ground floor.
- Room bathroom ("WC"), located in the International Building, ground floor.
- Room WC ("WC"), located in the International Building, ground floor.
- Room Men's Restroom ("Men's Restroom"), located in the International Building, Second Floor.
- Room Gents Room ("Gents Room"), located in the International Building, First Floor.
- Room Women's Restroom ("Women's Restroom"), located in the International Building, Second Floor.
- Room Ladies Room ("Ladies Room"), located in the International Building

{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.\n- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.\n- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.\n- Room Staff Restroom (WC) ("Staff Restroom (WC)"), located in the International Building, ground floor.\n- Room bathroom ("WC"), located in the International Building, ground floor.\n- Room WC ("WC"), located in the International Building, ground floor.\n- Room Men\'s Restroom ("Men\'s Restroom"), located in the International Building, Second Floor.\n- Room Gents Room ("Gents Room"), located in the International Building, First Floor.\n- Room Women\'s Restroom ("Women\'s Restroom"), located in the International Building, Second Floor.\n- Room Ladies Room ("Ladies Room"), located in the International Building, First Floor.',
 '

In [ ]:
ask("bathrooms")

[GENERAL]  (method: dataset-match (multiple: 17 results, itemized))
There is more than one match:

- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.
- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.
- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).
- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).
- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).
- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.
- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).
- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).
- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).
- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).
- Room bathroom ("WC"), locat

{'intent': 'general',
 'answer': 'There is more than one match:\n\n- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.\n- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.\n- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).\n- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).\n- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).\n- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.\n- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).\n- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).\n- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).\n- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).\n- Room bathroom ("WC"), located in the Internationa

In [ ]:
print(_expand_synonyms({"bathrooms"}))

{'restroom', 'lavatory', 'toilet', 'bathrooms', 'washroom'}


In [ ]:
ask("i need bathrooms")

[GENERAL]  (method: dataset-match (multiple: 17 results, itemized))
There is more than one match:

- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.
- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.
- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).
- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).
- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.
- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).
- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).
- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).
- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).
- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).
- Room Men's Restroom ("Men's

{'intent': 'general',
 'answer': 'There is more than one match:\n\n- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.\n- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.\n- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).\n- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).\n- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.\n- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).\n- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).\n- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).\n- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).\n- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).\n- Room Men\'s Restroom ("Men\'s Restroom"), located

In [ ]:
ask("i need bathrooms at innovation building")

[LOCATION]  (method: dataset-match (multiple: 7 results, itemized))
There is more than one match:

- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).
- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).
- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).
- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).
- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).
- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).
- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room bathroom ("WC"), located in the Innovation Building, level 0 (ground floor).\n- Room 206 ("WC 206"), located in the Innovation Building, level 2 (second floor).\n- Room 207 ("WC 207"), located in the Innovation Building, level 2 (second floor).\n- Room 306 ("WC 306"), located in the Innovation Building, level 3 (third floor).\n- Room 111 ("WC 111"), located in the Innovation Building, level 1 (first floor).\n- Room 110 ("WC 110"), located in the Innovation Building, level 1 (first floor).\n- Room 307 ("WC 307"), located in the Innovation Building, level 3 (third floor).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 7 results, itemized)'}

In [ ]:
ask("i need bathrooms at international building")

[LOCATION]  (method: dataset-match (multiple: 9 results, itemized))
There is more than one match:

- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.
- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.
- Room bathroom ("WC"), located in the International Building, ground floor.
- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.
- Room WC ("WC"), located in the International Building, ground floor.
- Room Staff Restroom (WC) ("Staff Restroom (WC)"), located in the International Building, ground floor.
- Room Men's Restroom ("Men's Restroom"), located in the International Building, Second Floor.
- Room Gents Room ("Gents Room"), located in the International Building, First Floor.
- Room Women's Restroom ("Women's Restroom"), located in the International Building, Second Floor.

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'There is more than one match:\n\n- Room Special Needs Bathroom ("Special Needs Bathroom"), located in the International Building, Underground.\n- Room Men Bathroom ("Men Bathroom"), located in the International Building, Underground.\n- Room bathroom ("WC"), located in the International Building, ground floor.\n- Room Women Bathroom ("Women Bathroom"), located in the International Building, Underground.\n- Room WC ("WC"), located in the International Building, ground floor.\n- Room Staff Restroom (WC) ("Staff Restroom (WC)"), located in the International Building, ground floor.\n- Room Men\'s Restroom ("Men\'s Restroom"), located in the International Building, Second Floor.\n- Room Gents Room ("Gents Room"), located in the International Building, First Floor.\n- Room Women\'s Restroom ("Women\'s Restroom"), located in the International Building, Second Floor.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'dataset-match (multiple: 9

In [ ]:
ask("i need tourism dep")

[GENERAL]  (method: faculty-match (Faculty of Tourism, score=1))
This document is about the Faculty of Tourism.

Faculty Name: Faculty of Tourism

Departments:
- Tourism and Hotel Management (Turkish Program)
- Tourism and Hotel Management (English Program)
- Gastronomy
- Cookery

Degree Types:
BSc, MSc

Website:
https://turizm.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'general',
 'answer': 'This document is about the Faculty of Tourism.\n\nFaculty Name: Faculty of Tourism\n\nDepartments:\n- Tourism and Hotel Management (Turkish Program)\n- Tourism and Hotel Management (English Program)\n- Gastronomy\n- Cookery\n\nDegree Types:\nBSc, MSc\n\nWebsite:\nhttps://turizm.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Tourism, score=1)'}

In [ ]:
print(sum(1 for d in all_docs if d.metadata.get('type') == 'hospital_info'))

1


In [ ]:
ask("Hospital Info")

[GENERAL]  (method: full-listing (hospital_info))
This document describes general information about NEU Hospital.
Total area: 55000 sqm
Blocks: East Block, West Block
Floors (East): Ground (0) through 4th floor
Floors (West): Ground (0) through 9th floor
Total patient rooms: 209
VIP rooms: 22
Max bed capacity: 500
Phone: +90 392 444 0 535
Website: https://neareasthospital.com
Address: Near East University Hospital, Near East Boulevard, Nicosia, North Cyprus
Appointment booking website: https://neareasthospital.com
Appointment booking phone: +90 392 444 0 535

Sources retrieved:
  • neu_hospital_departments.json


{'intent': 'general',
 'answer': 'This document describes general information about NEU Hospital.\nTotal area: 55000 sqm\nBlocks: East Block, West Block\nFloors (East): Ground (0) through 4th floor\nFloors (West): Ground (0) through 9th floor\nTotal patient rooms: 209\nVIP rooms: 22\nMax bed capacity: 500\nPhone: +90 392 444 0 535\nWebsite: https://neareasthospital.com\nAddress: Near East University Hospital, Near East Boulevard, Nicosia, North Cyprus\nAppointment booking website: https://neareasthospital.com\nAppointment booking phone: +90 392 444 0 535',
 'sources': 'Sources retrieved:\n  • neu_hospital_departments.json',
 'method': 'full-listing (hospital_info)'}

In [ ]:
q = "emregcney medicine"
search_query = _correct_query(q)
print("corrected:", search_query)

hits = vectorstore.similarity_search_with_score(search_query, k=10)
q_tokens = set(_tokenize(search_query))
for doc, l2_dist in hits:
    cos_sim = _cos_sim_from_l2(l2_dist)
    content_tokens = set(_tokenize(doc.metadata.get("question", doc.page_content)))
    shared = q_tokens & content_tokens
    print(f"type={doc.metadata.get('type'):20} cos_sim={cos_sim:.3f} shared={shared}")

corrected: emergency medicine


type=qa_pair              cos_sim=0.641 shared={'emergency'}
type=hospital_department  cos_sim=0.486 shared={'emergency', 'medicine'}
type=qa_pair              cos_sim=0.475 shared={'medicine'}
type=hospital_department  cos_sim=0.382 shared={'emergency'}
type=qa_pair              cos_sim=0.360 shared={'medicine'}
type=hospital_department  cos_sim=0.358 shared={'medicine'}
type=hospital_department  cos_sim=0.354 shared=set()
type=hospital_department  cos_sim=0.351 shared=set()
type=field                cos_sim=0.350 shared=set()
type=hospital_department  cos_sim=0.347 shared=set()


In [ ]:
result = dataset_answer("emregcney medicine")
print(result)

{'answer': 'This document describes a department at NEU Hospital.\n\nDepartment: Emergency Medicine\nFloor: 1st Floor\nBlock: West Block\nHours: Monday–Sunday: 24/7 – Open every day, all hours\nFacilities: Resuscitation room, Trauma room, Minor procedure room, 3 examination rooms, 4 triage rooms, 12-bed observation area with bedside monitors\n\nNotes: 24/7 emergency services including trauma, cardiac arrest, stroke; direct admission to ICU if needed', 'source': 'Emergency Medicine', 'method': 'dataset-match (chunk, score=0.49)', 'needs_phrasing': True}


In [ ]:
print(dataset_answer("where is hospital"))

{'answer': 'This document describes general information about NEU Hospital.\nTotal area: 55000 sqm\nBlocks: East Block, West Block\nFloors (East): Ground (0) through 4th floor\nFloors (West): Ground (0) through 9th floor\nTotal patient rooms: 209\nVIP rooms: 22\nMax bed capacity: 500\nPhone: +90 392 444 0 535\nWebsite: https://neareasthospital.com\nAddress: Near East University Hospital, Near East Boulevard, Nicosia, North Cyprus\nAppointment booking website: https://neareasthospital.com\nAppointment booking phone: +90 392 444 0 535', 'source': 'neu_hospital_departments.json', 'method': 'full-listing (hospital_info)', 'needs_phrasing': False}


In [ ]:
ask("where is emergency medicine department")

[LOCATION]  (method: faculty-match (Faculty of Medicine, score=1))
This document is about the Faculty of Medicine.

Faculty Name: Faculty of Medicine

Departments:
- Medicine

Degree Types:
MD, MSc, PhD

Website:
https://medicine.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'location',
 'answer': 'This document is about the Faculty of Medicine.\n\nFaculty Name: Faculty of Medicine\n\nDepartments:\n- Medicine\n\nDegree Types:\nMD, MSc, PhD\n\nWebsite:\nhttps://medicine.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Medicine, score=1)'}

In [ ]:
ask("where is emergency medicine")

[LOCATION]  (method: dataset-match (chunk, score=0.52))
Emergency Medicine is located on the 1st Floor, West Block.

Sources retrieved:
  • Emergency Medicine


{'intent': 'location',
 'answer': 'Emergency Medicine is located on the 1st Floor, West Block.',
 'sources': 'Sources retrieved:\n  • Emergency Medicine',
 'method': 'dataset-match (chunk, score=0.52)'}

In [ ]:
print([d.metadata.get("location_name") for d in all_docs
       if d.metadata.get("type") == "location"])

['Faculty of Engineering', 'Faculty of Medicine', 'Faculty of Dentistry / NEU Dental Hospital', 'Faculty of Architecture', 'Faculty of Law', 'Faculty of Economics and Administrative Sciences', 'Faculty of Pharmacy', 'Faculty of Nursing', 'Faculty of Arts and Sciences', 'Faculty of Communication', 'Faculty of Fine Arts and Design', 'Faculty of Tourism', 'Faculty of Civil and Environmental Engineering', 'Faculty of Artificial Intelligence and Informatics', 'Atatürk Faculty of Education', 'Faculty of Health Sciences', 'Faculty of Veterinary Medicine / Near East Animal Hospital', 'Faculty of Sports Sciences', 'Faculty of Agriculture', 'Faculty of Theology', 'Grand Library (Near East University Grand Library)', 'NEU Hospital (Near East Hospital)', 'Atatürk Culture and Congress Centre (AKKM)', 'Olympic Indoor Swimming Pool', 'Sports Tower', 'Student Dormitories', 'Post Office', 'Supermarket', 'Near East Bank (Campus Branch)', 'Türkiye İş Bankası (Campus Branch)', 'Ziraat Bankası ATM', 'Fuel 

In [ ]:
ask("how can i locate hospital")

In [ ]:
ask("how can i get to innovation building")

[LOCATION]  (method: llm-fallback (general knowledge))
To provide you with the best directions to the Innovation Building on the Near East University campus, could you please clarify which specific "Innovation Building" you are referring to? 

If you are looking for a specific research center or the NEU Innovation Center, please let me know, and I will be happy to guide you to its location on our campus map. Alternatively, if you are currently on campus, you can visit the main information desk at the Rectorate building, where our staff can provide you with a printed map and clear directions.


{'intent': 'location',
 'answer': 'To provide you with the best directions to the Innovation Building on the Near East University campus, could you please clarify which specific "Innovation Building" you are referring to? \n\nIf you are looking for a specific research center or the NEU Innovation Center, please let me know, and I will be happy to guide you to its location on our campus map. Alternatively, if you are currently on campus, you can visit the main information desk at the Rectorate building, where our staff can provide you with a printed map and clear directions.',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("how to get to innovation building ")

[LOCATION]  (method: llm-fallback (general knowledge))
To get to the Innovation Building at Near East University, you should head toward the main campus area in Nicosia. The building is situated within the university complex, near the central facilities.

If you are already on campus, you can follow the signs for the "Innovation and Information Technologies Center." If you are arriving from off-campus, you can reach the university via the main entrance; once you pass the security gates, follow the internal road signs pointing toward the Innovation Center or the central library area.

If you are having trouble finding it, I recommend checking the official NEU campus map available on our website or asking any of the campus security personnel, who are always happy to point you in the right direction!


{'intent': 'location',
 'answer': 'To get to the Innovation Building at Near East University, you should head toward the main campus area in Nicosia. The building is situated within the university complex, near the central facilities.\n\nIf you are already on campus, you can follow the signs for the "Innovation and Information Technologies Center." If you are arriving from off-campus, you can reach the university via the main entrance; once you pass the security gates, follow the internal road signs pointing toward the Innovation Center or the central library area.\n\nIf you are having trouble finding it, I recommend checking the official NEU campus map available on our website or asking any of the campus security personnel, who are always happy to point you in the right direction!',
 'sources': '',
 'method': 'llm-fallback (general knowledge)'}

In [ ]:
ask("where is Necibe")

[LOCATION]  (method: partial-lecturer-match (Necibe Can))
Necibe Can (Distance Education Coordination) is based in Room 115 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr Sezer Kanbul (Director), Prof. Dr. Fahriye Alt?nay (Al and Educational Technologies Coordinator), Zöhre Serttas, MSc. (E-Exam Coordinator), Doç. Dr. ?ahin Akda? (Hardware and Infrastructure Coordinator), Sinan Kudret, M.ed. (E-Exam Coordination), Zehra Parastou Sarshar (EBYS Coordinator), Mustafa Y?ld?r?m (E-Sinav Coordination), Prof. Dr. Zehra Alt?nay (Vice Director), Murat Cem Acaralp, MA. (Academic Journal and Publication
Coordinator), Pelin Gür Yar?msakall?, Msc. (Distance Education and Informatics
Coordinator), Vedia Ese, M.ed. (Distance Education Coordination), Aras.Gör. Hakan Bagmen (APDF, MYO Coordinator).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'Necibe Can (Distance Education Coordination) is based in Room 115 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr Sezer Kanbul (Director), Prof. Dr. Fahriye Alt?nay (Al and Educational Technologies Coordinator), Zöhre Serttas, MSc. (E-Exam Coordinator), Doç. Dr. ?ahin Akda? (Hardware and Infrastructure Coordinator), Sinan Kudret, M.ed. (E-Exam Coordination), Zehra Parastou Sarshar (EBYS Coordinator), Mustafa Y?ld?r?m (E-Sinav Coordination), Prof. Dr. Zehra Alt?nay (Vice Director), Murat Cem Acaralp, MA. (Academic Journal and Publication\nCoordinator), Pelin Gür Yar?msakall?, Msc. (Distance Education and Informatics\nCoordinator), Vedia Ese, M.ed. (Distance Education Coordination), Aras.Gör. Hakan Bagmen (APDF, MYO Coordinator).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'partial-lecturer-match (Necibe Can)'}

In [ ]:
ask("where is Necibe Can")            # still finds her
ask("where is Necibe")                # still finds her
ask("where is Pembe")                 # still finds her

[LOCATION]  (method: exact-lecturer-match (Necibe Can))
Necibe Can (Distance Education Coordination) is based in Room 115 ("Office"), located in the Innovation Building, level 1 (first floor). Also based there: Prof. Dr Sezer Kanbul (Director), Prof. Dr. Fahriye Alt?nay (Al and Educational Technologies Coordinator), Zöhre Serttas, MSc. (E-Exam Coordinator), Doç. Dr. ?ahin Akda? (Hardware and Infrastructure Coordinator), Sinan Kudret, M.ed. (E-Exam Coordination), Zehra Parastou Sarshar (EBYS Coordinator), Mustafa Y?ld?r?m (E-Sinav Coordination), Prof. Dr. Zehra Alt?nay (Vice Director), Murat Cem Acaralp, MA. (Academic Journal and Publication
Coordinator), Pelin Gür Yar?msakall?, Msc. (Distance Education and Informatics
Coordinator), Vedia Ese, M.ed. (Distance Education Coordination), Aras.Gör. Hakan Bagmen (APDF, MYO Coordinator).

Sources retrieved:
  • level1111.csv
[LOCATION]  (method: partial-lecturer-match (Necibe Can))
Necibe Can (Distance Education Coordination) is based in Room 

{'intent': 'location',
 'answer': 'Asst. Prof. Dr. Pembe Can (Assistant Professor) is based in Room Pembe Can Office ("Pembe Can Office"), located in the International Building, First Floor.',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'partial-lecturer-match (Asst. Prof. Dr. Pembe Can)'}

In [ ]:
ask("where is fadi")

[LOCATION]  (method: partial-lecturer-match (Prof. Dr. Fadi Alturjman, multiple rooms))
This name is associated with more than one location:

Prof. Dr. Fadi Alturjman (Faculty of Al and Informatics - Dean) is based in Room 303 ("Office"), located in the Innovation Building, level 3 (third floor). Also based there: Sinem Alturjman, MSc. (Artificial Intelligence Engineering).

---

Prof. Dr. Fadi Alturjman (AloT - Director) is based in Room 9 ("Office"), located in the Innovation Building, level 0 (ground floor). Also based there: Eng. Gresmarteneanos Tannous (AloT - Member), Eng. Nafiseh Yousefi Kian (AloT - Member), Eng. Farhaneh Aryanejad (AloT - Member), Sinem Alturjman, MSc. (AloT - Member), Zöhre Serttas, MSc. (AloT - Member), Precious Thom (AloT - Member).

Sources retrieved:
  • level1111.csv


{'intent': 'location',
 'answer': 'This name is associated with more than one location:\n\nProf. Dr. Fadi Alturjman (Faculty of Al and Informatics - Dean) is based in Room 303 ("Office"), located in the Innovation Building, level 3 (third floor). Also based there: Sinem Alturjman, MSc. (Artificial Intelligence Engineering).\n\n---\n\nProf. Dr. Fadi Alturjman (AloT - Director) is based in Room 9 ("Office"), located in the Innovation Building, level 0 (ground floor). Also based there: Eng. Gresmarteneanos Tannous (AloT - Member), Eng. Nafiseh Yousefi Kian (AloT - Member), Eng. Farhaneh Aryanejad (AloT - Member), Sinem Alturjman, MSc. (AloT - Member), Zöhre Serttas, MSc. (AloT - Member), Precious Thom (AloT - Member).',
 'sources': 'Sources retrieved:\n  • level1111.csv',
 'method': 'partial-lecturer-match (Prof. Dr. Fadi Alturjman, multiple rooms)'}

In [ ]:
ask("how many faculty in neu")

[ACADEMIC]  (method: faculty-count (20))
NEU has 20 faculties:

- Faculty of Architecture
- Atatürk Faculty of Education
- Faculty of Dentistry
- Faculty of Pharmacy
- Faculty of Arts and Sciences
- Faculty of Fine Arts and Design
- Faculty of Nursing
- Faculty of Law
- Faculty of Economics and Administrative Sciences
- Faculty of Civil and Environmental Engineering
- Faculty of Theology
- Faculty of Communication
- Faculty of Engineering
- Faculty of Health Sciences
- Faculty of Sports Sciences
- Faculty of Medicine
- Faculty of Tourism
- Faculty of Veterinary Medicine
- Faculty of Artificial Intelligence and Informatics
- Faculty of Agriculture

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'academic',
 'answer': 'NEU has 20 faculties:\n\n- Faculty of Architecture\n- Atatürk Faculty of Education\n- Faculty of Dentistry\n- Faculty of Pharmacy\n- Faculty of Arts and Sciences\n- Faculty of Fine Arts and Design\n- Faculty of Nursing\n- Faculty of Law\n- Faculty of Economics and Administrative Sciences\n- Faculty of Civil and Environmental Engineering\n- Faculty of Theology\n- Faculty of Communication\n- Faculty of Engineering\n- Faculty of Health Sciences\n- Faculty of Sports Sciences\n- Faculty of Medicine\n- Faculty of Tourism\n- Faculty of Veterinary Medicine\n- Faculty of Artificial Intelligence and Informatics\n- Faculty of Agriculture',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-count (20)'}

In [ ]:
ask("i need tourism departments")

  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: faculty-match (Faculty of Tourism, score=1))
This document is about the Faculty of Tourism.

Faculty Name: Faculty of Tourism

Departments:
- Tourism and Hotel Management (Turkish Program)
- Tourism and Hotel Management (English Program)
- Gastronomy
- Cookery

Degree Types:
BSc, MSc

Website:
https://turizm.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'academic',
 'answer': 'This document is about the Faculty of Tourism.\n\nFaculty Name: Faculty of Tourism\n\nDepartments:\n- Tourism and Hotel Management (Turkish Program)\n- Tourism and Hotel Management (English Program)\n- Gastronomy\n- Cookery\n\nDegree Types:\nBSc, MSc\n\nWebsite:\nhttps://turizm.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Tourism, score=1)'}

In [ ]:
ask("what departments are in the faculty of law")

  [spell-check] 'departments' → 'department'
[ACADEMIC]  (method: faculty-match (Faculty of Law, score=1))
This document is about the Faculty of Law.

Faculty Name: Faculty of Law

Departments:
- Law

Degree Types:
LLB, LLM, PhD

Website:
https://hukuk.neu.edu.tr

Sources retrieved:
  • neu_faculties_departments.json


{'intent': 'academic',
 'answer': 'This document is about the Faculty of Law.\n\nFaculty Name: Faculty of Law\n\nDepartments:\n- Law\n\nDegree Types:\nLLB, LLM, PhD\n\nWebsite:\nhttps://hukuk.neu.edu.tr',
 'sources': 'Sources retrieved:\n  • neu_faculties_departments.json',
 'method': 'faculty-match (Faculty of Law, score=1)'}